# 02 — Silver Pipeline

**Status: nine validated Silver datasets consumed by the implemented Gold model.**

This notebook explains the decisions and inspects their evidence. [The pipeline](../src/retail_pricing/pipeline.py) owns execution; [the Silver modules](../src/retail_pricing/) own every transformation.

Read each table's **grain → source issues → treatment → contract**, inspect the results, then skim its conclusion. The [README](../README.md) holds the longer rationale; [01 · Discovery](01_source_discovery.ipynb) retains the exploratory history.

Raw stays immutable. Missing attributes remain nullable, primary values win, and uncertainty stays visible. Natural keys remain in Silver; dimensional modelling belongs to Gold.


## 1. Load a Validated Checkpoint

Keep `RUN_PIPELINE = False` to inspect existing outputs. Set it to `True` to rebuild all nine outputs through `run_silver_pipeline()`; the same entry point is available from the CLI. Missing, changed or incomplete artifacts fail explicitly.

No database access is required. The optional environment paths support isolated validation; normal runs use the project's `data/raw` and `data/clean`.


In [1]:
# Locate the project and import the official pipeline, keeping transformations in src.
import os
from pathlib import Path

import duckdb
import polars as pl
from IPython.display import Markdown, display

from retail_pricing import SILVER_FILES
from retail_pricing.fmcg_taxonomy import classify_path
from retail_pricing.pipeline import OUTPUTS, load_silver_manifest, run_silver_pipeline
from retail_pricing.raw import validate_local_snapshot
from retail_pricing.reference import (
    ENCODED_SHOPS,
    add_shop_names,
    canonicalize_shop,
    get_nutrient_mapping,
    require_canonical_shops,
)

PROJECT_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").is_file() and (path / "src/retail_pricing").is_dir()
)
RAW_DIR = Path(os.environ.get("RETAIL_RAW_DIR", PROJECT_ROOT / "data/raw"))
SILVER_DIR = Path(os.environ.get("RETAIL_SILVER_DIR", PROJECT_ROOT / "data/clean"))
RUN_PIPELINE = False
pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(8)

polars.config.Config

In [2]:
# Build only on explicit request, then verify code, Raw lineage and all published files.
if RUN_PIPELINE:
    run_silver_pipeline(RAW_DIR, SILVER_DIR)
manifest = load_silver_manifest(SILVER_DIR)
raw_manifest = validate_local_snapshot(RAW_DIR)
if manifest["raw_files"] != raw_manifest["files"]:
    raise RuntimeError(
        "This Silver checkpoint does not correspond to the current Raw snapshot."
    )
metrics = manifest["metrics"]
pl.DataFrame(
    [
        {"dataset": f["dataset"], "rows": f["rows"], "file": f["file"]}
        for f in manifest["files"]
    ]
)

dataset,rows,file
str,i64,str
"""products""",32826,"""silver_products.parquet"""
"""locations""",1638,"""silver_locations.parquet"""
"""prices""",1198547,"""silver_prices.parquet"""
"""nutritionals""",9467978,"""silver_nutritionals.parquet"""
"""weekly_products""",114517,"""silver_weekly_products.parquet"""
"""weekly_locations""",1230,"""silver_weekly_locations.parque…"
"""weekly_prices""",19075099,"""silver_weekly_prices.parquet"""
"""product_category_paths""",154760,"""silver_product_category_paths.…"
"""product_primary_categories""",167077,"""silver_product_primary_categor…"


### 1.1 Reading the Evidence

Hard contracts stop publication on structural failure. DQ flags describe retained
exceptions. Metrics come from the published checkpoint or read-only queries.

| Shared treatment | Technical method | Effect / model use |
| --- | --- | --- |
| Canonical `shop` | Guarded string reversal over the complete reviewed source domain, implemented once for Polars and DuckDB. Canonical input is left unchanged; nulls stay null. | Decodes source identifiers; no ML model. |
| Reporting `shop_name` | Exact lookup from a version-controlled reference, added after business transformations. Gold consumes this persisted field. | Adds readable names; codes, keys and observations stay unchanged. `cg` → `Collect&Go` is a documented reviewed attribution; no ML model. |

See the [decoding evidence](../docs/silver-data-quality.md#retailer-codes-and-names) and
[name reference, examples and tests](../docs/silver-data-quality.md#retailer-codes-and-names).
Neither rule reverses arbitrary product text or infers country.


In [3]:
# Register lazy Parquet views and small display helpers without collecting the large facts.
duck = duckdb.connect()
source_names = {
    "weekly_products": "weekly_prices_products",
    "weekly_locations": "weekly_prices_locations",
}
for name in OUTPUTS:
    duck.read_parquet(str(SILVER_DIR / SILVER_FILES[name])).create_view(name)
    if name not in {"product_category_paths", "product_primary_categories"}:
        source = source_names.get(name, name)
        duck.read_parquet(str(RAW_DIR / f"{source}.parquet")).create_view(f"raw_{name}")


def show_metrics(name, keys=None):
    values = metrics[name]
    return pl.DataFrame(
        {
            "metric": list(keys or values),
            "value": [str(values[key]) for key in (keys or values)],
        }
    )


def missingness(raw, silver, columns):
    return pl.DataFrame(
        {
            "column": columns,
            "raw_nulls": [raw[c].null_count() for c in columns],
            "silver_nulls": [silver[c].null_count() for c in columns],
        }
    )


def converted_tokens(raw, silver, columns):
    frames = []
    for column in columns:
        converted = (
            raw.select("daltix_id", pl.col(column).alias("raw_value"))
            .join(
                silver.select("daltix_id", pl.col(column).alias("silver_value")),
                on="daltix_id",
                validate="1:1",
            )
            .filter(
                pl.col("raw_value").is_not_null() & pl.col("silver_value").is_null()
            )
        )
        frames.append(
            converted.group_by("raw_value")
            .len()
            .with_columns(pl.lit(column).alias("column"))
        )
    return (
        pl.concat(frames)
        .select("column", "raw_value", "len")
        .sort("column", "raw_value")
    )


def checkpoint_result(name):
    record = next(f for f in manifest["files"] if f["dataset"] == name)
    return {
        "dataset": name,
        "persisted_rows": record["rows"],
        "contracts": "passed before publication",
    }

### 1.2 Retailer codes: Raw → Silver

Every Raw source writes the `shop` code backwards (`idla` is `aldi`). One reviewed
decoder in [`reference.py`](../src/retail_pricing/reference.py) reverses the nine
observed codes at the Raw→Silver boundary; already-decoded codes pass through and any
new code stops the build for review. The table is the complete domain found across
all seven Raw sources, not a sample, with the Silver code the decoder produces.

In [4]:
raw_sources = {
    "products": "raw_products",
    "locations": "raw_locations",
    "prices": "raw_prices",
    "nutritionals": "raw_nutritionals",
    "weekly_products": "raw_weekly_products",
    "weekly_locations": "raw_weekly_locations",
    "weekly_prices": "raw_weekly_prices",
}
presence = pl.concat(
    [
        duck.sql(
            f"SELECT '{name}' AS raw_source, shop AS raw_code, count(*) AS raw_rows "
            f"FROM {view} GROUP BY shop"
        ).pl()
        for name, view in raw_sources.items()
    ]
)
decoded = canonicalize_shop(pl.DataFrame({"shop": list(ENCODED_SHOPS)}))
decoding = (
    pl.DataFrame({"raw_code": list(ENCODED_SHOPS), "silver_shop": decoded["shop"]})
    .pipe(lambda frame: add_shop_names(frame.rename({"silver_shop": "shop"})))
    .rename({"shop": "silver_shop"})
    .join(
        presence.group_by("raw_code").agg(
            pl.col("raw_source").sort().alias("raw_sources_using_code"),
            pl.col("raw_rows").sum().alias("raw_rows"),
        ),
        on="raw_code",
        how="left",
    )
    .sort("silver_shop")
)
with pl.Config(tbl_rows=12, tbl_cols=6, fmt_table_cell_list_len=8):
    display(decoding)

raw_code,silver_shop,shop_name,raw_sources_using_code,raw_rows
str,str,str,list[str],i64
"""ha""","""ah""","""Albert Heijn""","[""locations"", ""nutritionals"", ""products""]",774081
"""idla""","""aldi""","""Aldi""","[""locations"", ""products"", ""weekly_locations"", ""weekly_prices"", ""weekly_products""]",183670
"""gc""","""cg""","""Collect&Go""","[""locations"", ""nutritionals"", ""prices"", ""products""]",70372
"""plc""","""clp""","""Colruyt Lowest Prices""","[""locations"", ""nutritionals"", ""prices"", ""weekly_locations"", ""weekly_prices"", ""weekly_products""]",5722752
"""frc""","""crf""","""Carrefour""","[""locations"", ""nutritionals"", ""prices"", ""products"", ""weekly_locations"", ""weekly_prices"", ""weekly_products""]",13188651
"""lld""","""dll""","""Delhaize""","[""locations"", ""nutritionals"", ""prices"", ""products"", ""weekly_locations"", ""weekly_prices"", ""weekly_products""]",1666428
"""obmuj""","""jumbo""","""Jumbo""","[""locations"", ""nutritionals"", ""prices"", ""products""]",56888
"""ldil""","""lidl""","""Lidl""","[""locations"", ""products""]",528
"""raps""","""spar""","""SPAR""","[""locations""]",1


In [5]:
# The decoded domain is the same in every published Silver table.
silver_shops = pl.concat(
    [
        duck.sql(
            f"SELECT '{name}' AS dataset, shop, count(*) AS rows FROM {name} GROUP BY shop"
        ).pl()
        for name in OUTPUTS
    ]
)
require_canonical_shops(silver_shops["shop"].unique().to_list())
silver_shops.pivot(on="shop", index="dataset", values="rows").sort("dataset")

dataset,cg,crf,dll,…,aldi,ah,clp,spar
str,i64,i64,i64,…,i64,i64,i64,i64
"""locations""",41,47,749,…,392,55,42,1
"""nutritionals""",127327,2208485,127968,…,null,6677470,236381,null
"""prices""",49947,532827,4012,…,null,null,572469,null
"""product_category_paths""",5812,68358,36404,…,11622,10643,16506,null
"""product_primary_categories""",9871,46759,33595,…,14069,23652,24451,null
"""products""",5803,6327,3900,…,1345,9889,null,null
"""weekly_locations""",null,47,749,…,392,null,42,null
"""weekly_prices""",null,12249426,1596665,…,167039,null,5061969,null
"""weekly_products""",null,44391,33369,…,13166,null,23591,null


### 1.3 Retailer names

Names are a separate reviewed attribute, `shop_name`, so codes stay the stable keys
and reports read *Carrefour* instead of `crf`. Six names are direct readings of the
decoded code; `dll`, `clp`, `crf` and `ah` rest on Raw product text and store formats.
`cg` → **Collect&Go** is the one reviewed attribution that no code book confirms:
Collect&Go text appears under `clp`, while Colruyt assortment and store overlap
support the label, so it is documented rather than presented as fact. Every Silver
table gets the name after its business rules, and an unknown code fails.

In [6]:
duck.sql(
    "SELECT shop, shop_name, count(DISTINCT dataset) AS silver_datasets "
    "FROM (SELECT 'weekly_prices' AS dataset, shop, shop_name FROM weekly_prices "
    "UNION ALL SELECT 'products', shop, shop_name FROM products "
    "UNION ALL SELECT 'locations', shop, shop_name FROM locations "
    "UNION ALL SELECT 'prices', shop, shop_name FROM prices "
    "UNION ALL SELECT 'nutritionals', shop, shop_name FROM nutritionals "
    "UNION ALL SELECT 'weekly_products', shop, shop_name FROM weekly_products "
    "UNION ALL SELECT 'weekly_locations', shop, shop_name FROM weekly_locations) "
    "GROUP BY ALL ORDER BY shop"
).pl()

shop,shop_name,silver_datasets
str,str,i64
"""ah""","""Albert Heijn""",3
"""aldi""","""Aldi""",5
"""cg""","""Collect&Go""",4
"""clp""","""Colruyt Lowest Prices""",6
"""crf""","""Carrefour""",7
"""dll""","""Delhaize""",7
"""jumbo""","""Jumbo""",4
"""lidl""","""Lidl""",2
"""spar""","""SPAR""",1


## 2. Products

**Purpose / grain:** a nullable product reference; one row per `daltix_id` within this source.

**Issues → treatment:** normalize confirmed placeholders, retain country/language, and preserve descriptive gaps.

- Common missing tokens (`""`, `null`, `none`, `n/a`, `#N/A`) become NULL in every text column.
- Three descriptive columns also carry their **own placeholders**, found by counting whole-value matches in Raw: `name` (`not provided`), `brand` (`not provided`, `no brand`, `unknown`, `*`, `.`, `geen`) and `description` (`x`, `*`, `0`, `-`, `.`). They become NULL, but only when they are the entire value: `x` is a placeholder description, `*mag in vaatwasser` is a real one.
- `NAN` is a real brand, not a missing token.

A NULL is honest: a name of `not provided` would otherwise be counted as a product called "not provided" in every count of names, brands and descriptions.

**Contract:** required source schema, preserved row count, complete source context and unique ID. [Implementation](../src/retail_pricing/silver_products.py) and [`quality.py`](../src/retail_pricing/quality.py).

In [7]:
# Inspect the source reference before comparing it with the published result.
products_raw = pl.read_parquet(RAW_DIR / "products.parquet")
products_raw.head()

daltix_id,shop,name,brand,country,description,categories,language
str,str,str,str,str,str,str,str
"""a3ed88636b0d973d4cd419c19a0116…","""idla""","""PALAZZO® Sauce pâtes rouge""","""palazzo®""","""lu""","""choix entre sauce légumes gril…","""[ [ ""Produits"", ""Ass…","""fr"""
"""c38129dc388664ce48e060eebb9598…","""frc""","""Axe BodySpray Deodorant You 15…","""axe""","""be""",null,"""[ [ ""Verzorging & hygiën…","""nl"""
"""15fbf0a66ff12e20fde41913aebd72…","""idla""","""ALL SEASONS® Fijnproeverssla""","""all seasons®""","""be""","""gebruiksklare mengeling van kr…","""[ [ ""Producten"", ""As…","""nl"""
"""7c93bb76b2772b001b264b779cb60c…","""ha""","""Blistex Lip relief crème""","""blistex""","""nl""","""lip relief cream helpt bij pij…","""[ [ ""Baby, verzorging en…","""nl"""
"""d60208deb02f76de96a8f771b1652d…","""frc""","""Cartec Achteruitkijkspiegel me…","""cartec""","""be""","""""","""[ [ ""Niet-voeding"" ], …","""nl"""


In [8]:
# Read the official product output, keeping the notebook free of cleaning rules.
products = pl.read_parquet(SILVER_DIR / SILVER_FILES["products"])
products.head()

daltix_id,shop,name,brand,…,description,categories,language,shop_name
str,str,str,str,…,str,str,str,str
"""a3ed88636b0d973d4cd419c19a0116…","""aldi""","""PALAZZO® Sauce pâtes rouge""","""palazzo®""",…,"""choix entre sauce légumes gril…","""[ [ ""Produits"", ""Ass…","""fr""","""Aldi"""
"""c38129dc388664ce48e060eebb9598…","""crf""","""Axe BodySpray Deodorant You 15…","""axe""",…,null,"""[ [ ""Verzorging & hygiën…","""nl""","""Carrefour"""
"""15fbf0a66ff12e20fde41913aebd72…","""aldi""","""ALL SEASONS® Fijnproeverssla""","""all seasons®""",…,"""gebruiksklare mengeling van kr…","""[ [ ""Producten"", ""As…","""nl""","""Aldi"""
"""7c93bb76b2772b001b264b779cb60c…","""ah""","""Blistex Lip relief crème""","""blistex""",…,"""lip relief cream helpt bij pij…","""[ [ ""Baby, verzorging en…","""nl""","""Albert Heijn"""
"""d60208deb02f76de96a8f771b1652d…","""crf""","""Cartec Achteruitkijkspiegel me…","""cartec""",…,null,"""[ [ ""Niet-voeding"" ], …","""nl""","""Carrefour"""


In [9]:
# Separate literal Raw nulls from the assessed missingness represented in Silver.
attributes = ["name", "brand", "description", "categories"]
missingness(products_raw, products, attributes)

column,raw_nulls,silver_nulls
str,i64,i64
"""name""",41,42
"""brand""",740,1387
"""description""",3393,8292
"""categories""",497,497


In [10]:
# List the actual source tokens converted to null so information loss is visible.
product_tokens = converted_tokens(products_raw, products, attributes)
product_tokens

column,raw_value,len
str,str,u32
"""brand""","""""",539
"""brand""","""*""",4
"""brand""",""".""",15
"""brand""","""geen""",1
"""brand""","""no brand""",82
"""brand""","""unknown""",6
"""description""","""""",4620
"""description""","""*""",45
"""description""","""0""",3


In [11]:
# Retain the description-token check and confirm that legitimate NAN brands survive.
display(product_tokens.filter(pl.col("column") == "description"))
products.filter(pl.col("brand").str.to_lowercase() == "nan").select(
    "daltix_id", "name", "brand"
)

column,raw_value,len
str,str,u32
"""description""","""""",4620
"""description""","""*""",45
"""description""","""0""",3
"""description""","""x""",231


daltix_id,name,brand
str,str,str
"""56fa6176fccc2cf68ca95d6e6ae09e…","""NAN Optipro Groeimelk 1+ vanaf…","""nan"""
"""febed5d0fd1968156b23d4ae5d3af2…","""NAN Optipro Groeimelk 1+ vanaf…","""nan"""


In [12]:
# Show key integrity and the persisted contract checkpoint alongside the missingness metrics.
display(
    products.select(
        pl.len().alias("rows"), pl.col("daltix_id").n_unique().alias("business_keys")
    )
)
display(show_metrics("products"))
checkpoint_result("products")

rows,business_keys
u32,u32
32826,32826


metric,value
str,str
"""rows""","""32826"""
"""missing_name""","""42"""
"""missing_brand""","""1387"""
"""missing_description""","""8292"""
"""missing_categories""","""497"""


{'dataset': 'products',
 'persisted_rows': 32826,
 'contracts': 'passed before publication'}

In [13]:
# The conclusion is computed from the published dataset.
display(
    Markdown(
        "### 2.1 Conclusion\n\n"
        f"**Established:** {products.height:,} product rows; the business key remains unique.\n\n"
        f"**Caveats:** {products['name'].null_count():,} names, {products['brand'].null_count():,} brands, "
        f"{products['description'].null_count():,} descriptions and {products['categories'].null_count():,} "
        "categories remain missing (placeholders included).\n\n"
        "**Silver decision:** Keep optional attributes nullable. Use this source for compatible "
        "fallback only; do not infer universal ID or historical validity."
    )
)

### 2.1 Conclusion

**Established:** 32,826 product rows; the business key remains unique.

**Caveats:** 42 names, 1,387 brands, 8,292 descriptions and 497 categories remain missing (placeholders included).

**Silver decision:** Keep optional attributes nullable. Use this source for compatible fallback only; do not infer universal ID or historical validity.

## 3. Locations

**Purpose / grain:** one location within a retailer; working key `(shop, id)`.

**Issues → treatment:** preserve both records in the known collision, mark them unsafe for enrichment, and omit the entirely empty `type`. Keep nullable postcode/coordinates and explicit country.

**Contract:** row preservation, complete key/context, finite in-range coordinates when present and consistent coordinate pairs. Uniqueness is **not** forced. [Implementation](../src/retail_pricing/silver_locations.py).


In [14]:
# Inspect Raw fields, including the empty type column retained in the immutable source.
duck.sql("SELECT * FROM raw_locations LIMIT 5").pl()

shop,country_code,id,type,geolocation_latitude,geolocation_longitude,postcode,sources
str,str,str,str,f64,f64,str,str
"""ldil""","""nl""","""1a131""",null,null,null,null,"""[ ""online"" ]"""
"""ldil""","""de""","""5f02f""",null,null,null,null,"""[ ""online"" ]"""
"""gc""","""be""","""f02af""",null,51.015406,3.6528982,"""9051""","""[ ""online"" ]"""
"""frc""","""be""","""64003""",null,50.908945,4.3048594,"""1780""","""[ ""online"" ]"""
"""frc""","""be""","""dd6cf""",null,50.492171,3.9675296,"""7010""","""[ ""online"" ]"""


In [15]:
# Inspect the cleaned reference and its retained business keys.
locations = pl.read_parquet(SILVER_DIR / SILVER_FILES["locations"])
locations.head()

shop,country_code,id,geolocation_latitude,…,dq_invalid_longitude,dq_incomplete_coordinates,dq_outside_benelux,shop_name
str,str,str,f64,…,bool,bool,bool,str
"""lidl""","""nl""","""1a131""",null,…,false,false,false,"""Lidl"""
"""lidl""","""de""","""5f02f""",null,…,false,false,false,"""Lidl"""
"""cg""","""be""","""f02af""",51.015406,…,false,false,false,"""Collect&Go"""
"""crf""","""be""","""64003""",50.908945,…,false,false,false,"""Carrefour"""
"""crf""","""be""","""dd6cf""",50.492171,…,false,false,false,"""Carrefour"""


In [16]:
# Keep the two conflicting location records visible rather than choosing an arbitrary winner.
locations.filter(pl.col("dq_business_key_collision"))

shop,country_code,id,geolocation_latitude,…,dq_invalid_longitude,dq_incomplete_coordinates,dq_outside_benelux,shop_name
str,str,str,f64,…,bool,bool,bool,str
"""dll""","""be""","""f334d""",50.601871,…,false,false,false,"""Delhaize"""
"""dll""","""be""","""f334d""",50.452781,…,false,false,false,"""Delhaize"""


In [17]:
# Inspect geographic flags on the persisted output.
locations.select(
    "shop",
    "id",
    "dq_invalid_latitude",
    "dq_invalid_longitude",
    "dq_incomplete_coordinates",
).head()

shop,id,dq_invalid_latitude,dq_invalid_longitude,dq_incomplete_coordinates
str,str,bool,bool,bool
"""lidl""","""1a131""",false,false,false
"""lidl""","""5f02f""",false,false,false
"""cg""","""f02af""",false,false,false
"""crf""","""64003""",false,false,false
"""crf""","""dd6cf""",false,false,false


In [18]:
# Measure retained gaps and geographic validity without treating missing as invalid.
display(show_metrics("locations"))
locations.select(
    pl.col("dq_incomplete_coordinates").sum().alias("incomplete_coordinate_rows")
)

metric,value
str,str
"""rows""","""1638"""
"""business_key_collision_rows""","""2"""
"""missing_postcode""","""40"""
"""missing_coordinates""","""18"""
"""invalid_latitudes""","""0"""
"""invalid_longitudes""","""0"""
"""outside_benelux""","""1"""


incomplete_coordinate_rows
u32
0


In [19]:
# Show why the source key is a working key rather than a global uniqueness contract.
display(
    locations.select(
        pl.len().alias("rows"),
        pl.struct("shop", "id").n_unique().alias("business_keys"),
    )
)
checkpoint_result("locations")

rows,business_keys
u32,u32
1638,1637


{'dataset': 'locations',
 'persisted_rows': 1638,
 'contracts': 'passed before publication'}

### 3.1 Conclusion

**Established:** 1,638 rows and 1,637 shop/ID keys; both `dll + f334d` (Raw `lld`) records are retained.

**Caveats:** 40 missing postcodes and 18 missing coordinate pairs; no out-of-range coordinates.

**Silver decision:** Exclude ambiguous keys from automatic fallback. Keep country explicit and preserve nullable geography.


## 4. Prices

**Purpose / grain:** a collected price at `(daltix_id, shop, location, downloaded_on)`.

**Issues → treatment:** retain meaningful null promotional prices. The adopted source convention interprets null as no active promotion; present values below regular price set `is_promotion`. This convention is source-specific.

**Contract:** exact key uniqueness, complete required fields, finite positive prices and preserved rows. Invalid numeric text must fail, not silently become null. [Implementation](../src/retail_pricing/silver_prices.py).


In [20]:
# Inspect Raw prices without loading the complete fact into Python.
duck.sql("SELECT * FROM raw_prices LIMIT 5").pl()

daltix_id,shop,country,location,…,promo_price,unit_std,currency,downloaded_on
str,str,str,str,…,f64,str,str,date
"""40f23e8eddf2cd9f28b1c50e5d856c…","""frc""","""be""","""f2177""",…,null,"""su""","""eur""",2020-04-17
"""928fe14059bb99a4ed1613b39f887f…","""frc""","""be""","""9ae3b""",…,null,"""su""","""eur""",2020-04-17
"""1f004043ca1559c432904afa20db08…","""frc""","""be""","""dd6cf""",…,null,"""su""","""eur""",2020-04-17
"""77eb025c8484adb9bda90be7f8c657…","""frc""","""be""","""7f6c1""",…,null,"""su""","""eur""",2020-04-17
"""28c1a2ccaa516fb33c438b23795631…","""frc""","""be""","""77446""",…,null,"""su""","""eur""",2020-04-17


In [21]:
# Inspect the official typed Silver output.
duck.sql("SELECT * FROM prices LIMIT 5").pl()

daltix_id,shop,country,location,…,dq_non_positive_price,dq_non_positive_promo_price,dq_promo_not_below_price,shop_name
str,str,str,str,…,bool,bool,bool,str
"""40f23e8eddf2cd9f28b1c50e5d856c…","""crf""","""be""","""f2177""",…,false,false,false,"""Carrefour"""
"""928fe14059bb99a4ed1613b39f887f…","""crf""","""be""","""9ae3b""",…,false,false,false,"""Carrefour"""
"""1f004043ca1559c432904afa20db08…","""crf""","""be""","""dd6cf""",…,false,false,false,"""Carrefour"""
"""77eb025c8484adb9bda90be7f8c657…","""crf""","""be""","""7f6c1""",…,false,false,false,"""Carrefour"""
"""28c1a2ccaa516fb33c438b23795631…","""crf""","""be""","""77446""",…,false,false,false,"""Carrefour"""


In [22]:
# Inspect promotion and price-quality flags independently of the transformation.
duck.sql(
    "SELECT price, promo_price, is_promotion, dq_non_positive_price, dq_non_positive_promo_price, dq_promo_not_below_price FROM prices LIMIT 5"
).pl()

price,promo_price,is_promotion,dq_non_positive_price,dq_non_positive_promo_price,dq_promo_not_below_price
f64,f64,bool,bool,bool,bool
3.025,null,false,false,false,false
2.915,null,false,false,false,false
1.298,null,false,false,false,false
2.145,null,false,false,false,false
2.959,null,false,false,false,false


In [23]:
# Retain the exact grain check as visible evidence.
duck.sql(
    "SELECT count(*) AS rows, count(DISTINCT (daltix_id, shop, location, downloaded_on)) AS business_keys FROM prices"
).pl()

rows,business_keys
i64,i64
1198547,1198547


In [24]:
# Report promotion metrics, units and date coverage from the persisted data.
display(show_metrics("prices"))
duck.sql("""
    SELECT count(DISTINCT unit_std) AS distinct_units, count(DISTINCT currency) AS distinct_currencies,
        min(downloaded_on) AS first_date, max(downloaded_on) AS last_date,
        count(*) FILTER (WHERE dq_non_positive_price) AS non_positive_prices,
        count(*) FILTER (WHERE dq_non_positive_promo_price) AS non_positive_promos
    FROM prices
""").pl()

metric,value
str,str
"""rows""","""1198547"""
"""rows_without_promo_price""","""1191783"""
"""promotion_rows""","""6764"""
"""unexpected_promo_rows""","""0"""
"""distinct_products""","""116"""
"""distinct_locations""","""136"""


distinct_units,distinct_currencies,first_date,last_date,non_positive_prices,non_positive_promos
i64,i64,date,date,i64,i64
1,1,2020-02-25,2021-02-25,0,0


In [25]:
# Confirm that the displayed data belongs to a completed publication.
checkpoint_result("prices")

{'dataset': 'prices',
 'persisted_rows': 1198547,
 'contracts': 'passed before publication'}

### 4.1 Conclusion

**Established:** 1,198,547 rows; 6,764 promotions and no unexpected populated promotional prices.

**Caveats:** 1,191,783 promotional prices remain null; the source convention is an adopted interpretation, not proof from null frequency alone.

**Silver decision:** Keep nulls, explicit unit/currency and the full observation key. Do not harmonize with weekly promotion semantics by simple coalescing.


## 5. Nutritionals

**Purpose / grain:** long nutrient observations at `(daltix_id, shop, country, download_date, nutrient_name)`.

**Issues → treatment:** retain completeness-first version selection and its conflict evidence. Standardize the 20 observed nutrient/unit pairs through the [official mapping](../src/retail_pricing/reference.py). Preserve all three source fields and the selected payload hash.

**Contract:** valid numeric source values, explicit mapping coverage, one canonical unit per nutrient, unique long grain and row reconciliation. Only known incompatible units produce canonical `NULL`; unknown pairs stop publication. [Implementation](../src/retail_pricing/silver_nutrition.py).


In [26]:
# Inspect the original nested source and its schema before reading the normalized output.
display(duck.sql("DESCRIBE raw_nutritionals").pl())
duck.sql("SELECT * FROM raw_nutritionals LIMIT 5").pl()

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""daltix_id""","""VARCHAR""","""YES""",null,null,null
"""shop""","""VARCHAR""","""YES""",null,null,null
"""country""","""VARCHAR""","""YES""",null,null,null
"""download_date""","""DATE""","""YES""",null,null,null
"""nutritional_values_std""","""VARCHAR""","""YES""",null,null,null
"""language""","""VARCHAR""","""YES""",null,null,null


daltix_id,shop,country,download_date,nutritional_values_std,language
str,str,str,date,str,str
"""8d93eace1a61d0644680915e05ce88…","""frc""","""be""",2021-01-28,"""{ ""nutrients"": { ""carboh…","""nl"""
"""b13ea65799dc89cf764c8759082d25…","""frc""","""be""",2021-01-28,"""{ ""nutrients"": { ""carboh…","""nl"""
"""447fa714c27770edae98dcf63c31c5…","""frc""","""be""",2021-01-28,"""{ ""nutrients"": { ""carboh…","""nl"""
"""327ea94401fa7b2eee762b710f08bc…","""frc""","""be""",2021-01-28,"""{ ""nutrients"": { ""carboh…","""nl"""
"""a4a10f28bc35e883d6caafff75631a…","""lld""","""be""",2021-01-27,"""{ ""nutrients"": { ""carboh…","""nl"""


In [27]:
# Inspect the selected source context without recomputing canonicalization.
duck.sql(
    "SELECT DISTINCT daltix_id, shop, country, language, download_date FROM nutritionals LIMIT 5"
).pl()

daltix_id,shop,country,language,download_date
str,str,str,str,date
"""181ea605006dd1a099a1f51923eb17…","""ah""","""nl""","""nl""",2020-12-05
"""181ecc2a6c3618da7c693d380bcd42…","""ah""","""nl""","""nl""",2020-12-22
"""1822f2c2965ebf0f0814bd56594e2c…","""ah""","""be""","""nl""",2020-12-01
"""1822f2c2965ebf0f0814bd56594e2c…","""ah""","""be""","""nl""",2020-12-30
"""182e2dce22e58e229a65d293acca05…","""jumbo""","""nl""","""nl""",2021-02-06


In [28]:
# Retain the Raw JSON validity and structure evidence alongside the stronger pipeline contract.
duck.sql("""
    SELECT count(*) AS rows,
        count(*) FILTER (WHERE nutritional_values_std IS NULL) AS missing_json_rows,
        count(*) FILTER (WHERE NOT json_valid(nutritional_values_std)) AS invalid_json_rows,
        count(*) FILTER (WHERE json_type(nutritional_values_std, '$.nutrients') IS DISTINCT FROM 'OBJECT') AS invalid_nutrient_objects
    FROM raw_nutritionals
""").pl()

rows,missing_json_rows,invalid_json_rows,invalid_nutrient_objects
i64,i64,i64,i64
1096542,0,0,0


In [29]:
# Profile every Raw nutrient/unit pair before selection so unsupported combinations cannot be hidden.
with pl.Config(tbl_rows=25, tbl_cols=6):
    display(
        duck.sql("""
        SELECT nutrient.key AS nutrient_name,
            json_extract_string(nutrient.value, '$.unit') AS source_unit,
            count(*) AS raw_rows, count(DISTINCT daltix_id) AS products,
            min(try_cast(json_extract_string(nutrient.value, '$.value') AS DOUBLE)) AS min_value,
            max(try_cast(json_extract_string(nutrient.value, '$.value') AS DOUBLE)) AS max_value
        FROM raw_nutritionals r
        CROSS JOIN LATERAL json_each(r.nutritional_values_std, '$.nutrients') nutrient
        GROUP BY ALL ORDER BY nutrient_name, source_unit
    """).pl()
    )

nutrient_name,source_unit,raw_rows,products,min_value,max_value
str,str,i64,i64,f64,f64
"""carbohydrates""","""g""",1086003,51787,0.0,986.666667
"""energy""","""g""",30,2,100.0,1032.0
"""energy""","""kJ""",1081972,49669,0.0,43666.666667
"""fats""","""g""",1080355,53152,0.0,741.65
"""fats""","""kJ""",27,1,962.32,962.32
"""fibers""","""g""",764133,34308,0.0,180.0
"""kilocalories""","""kcal""",1082002,49671,0.0,10436.583811
"""monounsaturated_fats""","""g""",9213,1027,0.0,310.4
"""polyols""","""g""",20998,949,0.0,99.7


In [30]:
# Report source deduplication and canonical observation counts from the official run.
show_metrics(
    "nutritionals",
    [
        "exact_duplicates_removed",
        "canonical_observations",
        "resolved_repeated_grains",
        "resolved_nutritional_conflicts",
        "resolved_language_conflicts",
        "hash_tiebreak_grains",
    ],
)

metric,value
str,str
"""exact_duplicates_removed""","""0"""
"""canonical_observations""","""1096442"""
"""resolved_repeated_grains""","""100"""
"""resolved_nutritional_conflicts""","""100"""
"""resolved_language_conflicts""","""0"""
"""hash_tiebreak_grains""","""91"""


In [31]:
# Inspect the selected versions and the provenance explaining each choice.
duck.sql("""
    SELECT DISTINCT daltix_id, shop, country, download_date, source_payload_hash,
        numeric_nutrient_count, nutrient_count, populated_unit_count, nutrition_selection_reason
    FROM nutritionals WHERE dq_repeated_grain
    ORDER BY daltix_id, shop, country, download_date LIMIT 10
""").pl()

daltix_id,shop,country,download_date,…,numeric_nutrient_count,nutrient_count,populated_unit_count,nutrition_selection_reason
str,str,str,date,…,i64,i64,i64,str
"""05766389442155ce47fa08689d547c…","""ah""","""nl""",2020-12-18,…,9,9,9,"""deterministic_hash_tiebreak"""
"""08e74e5c36f50fde38757429acf3bb…","""crf""","""be""",2021-02-02,…,9,9,9,"""completeness_first"""
"""17c65f953f988a273400c335cd7e73…","""ah""","""nl""",2020-12-29,…,9,9,9,"""deterministic_hash_tiebreak"""
"""17d33e7b62170c7f2d957e0b365316…","""crf""","""be""",2021-02-04,…,9,9,9,"""deterministic_hash_tiebreak"""
"""182ab99e8d1c6f56126dda5d9c3d8e…","""ah""","""nl""",2020-11-27,…,9,9,9,"""deterministic_hash_tiebreak"""
"""18a94ff31934e33d3df52eae9cd195…","""crf""","""be""",2021-02-08,…,9,9,9,"""deterministic_hash_tiebreak"""
"""199328fdfdf7b3385c951fc265aa48…","""crf""","""be""",2021-02-05,…,8,8,8,"""deterministic_hash_tiebreak"""
"""1a112b3283f106e1f7ea6c77307bed…","""ah""","""nl""",2020-12-04,…,9,9,9,"""deterministic_hash_tiebreak"""
"""1b342df565cbb5dee3600c3e54eb7c…","""ah""","""be""",2020-12-29,…,9,9,9,"""deterministic_hash_tiebreak"""


In [32]:
# Separate genuine completeness wins from deterministic ties instead of calling every choice resolved.
duck.sql("""
    SELECT nutrition_selection_reason,
        count(DISTINCT (daltix_id, shop, country, download_date)) AS observations
    FROM nutritionals GROUP BY nutrition_selection_reason ORDER BY observations DESC
""").pl()

nutrition_selection_reason,observations
str,i64
"""single_observation""",1096342
"""deterministic_hash_tiebreak""",91
"""completeness_first""",9


**Unit policy.** Fourteen pairs retain their values with factor 1. Six incompatible pairs retain their Raw values but have no conversion factor or canonical value. The canonical unit is the expected unit, not evidence that the original value was convertible. `energy` and `kilocalories` remain separate measures; no nutrient aliases were observed.


In [33]:
# Display the exact reference used by src; no second mapping is defined in this notebook.
with pl.Config(tbl_rows=25, tbl_cols=6):
    display(get_nutrient_mapping())

raw_nutrient_name,raw_unit,canonical_nutrient_name,canonical_unit,conversion_factor,mapping_status
str,str,str,str,f64,str
"""carbohydrates""","""g""","""carbohydrates""","""g""",1.0,"""identity"""
"""energy""","""kJ""","""energy""","""kJ""",1.0,"""identity"""
"""fats""","""g""","""fats""","""g""",1.0,"""identity"""
"""fibers""","""g""","""fibers""","""g""",1.0,"""identity"""
"""kilocalories""","""kcal""","""kilocalories""","""kcal""",1.0,"""identity"""
"""monounsaturated_fats""","""g""","""monounsaturated_fats""","""g""",1.0,"""identity"""
"""polyols""","""g""","""polyols""","""g""",1.0,"""identity"""
"""polyunsaturated_fats""","""g""","""polyunsaturated_fats""","""g""",1.0,"""identity"""
"""proteins""","""g""","""proteins""","""g""",1.0,"""identity"""


In [34]:
# Inspect source and canonical representations with the portion needed to interpret each value.
duck.sql("""
    SELECT daltix_id, download_date, nutrient_name_raw, nutrient_value_raw, nutrient_unit_raw,
        nutrient_name, nutrient_value, nutrient_unit, portion_value, portion_unit
    FROM nutritionals ORDER BY daltix_id, download_date, nutrient_name LIMIT 15
""").pl()

daltix_id,download_date,nutrient_name_raw,nutrient_value_raw,…,nutrient_value,nutrient_unit,portion_value,portion_unit
str,date,str,str,…,f64,str,f64,str
"""0002c6e437ae0c4f3a9129ba13a702…",2020-11-27,"""carbohydrates""","""0""",…,0.0,"""g""",100.0,"""g"""
"""0002c6e437ae0c4f3a9129ba13a702…",2020-11-27,"""energy""","""1066.92""",…,1066.92,"""kJ""",100.0,"""g"""
"""0002c6e437ae0c4f3a9129ba13a702…",2020-11-27,"""fats""","""13""",…,13.0,"""g""",100.0,"""g"""
"""0002c6e437ae0c4f3a9129ba13a702…",2020-11-27,"""fibers""","""0""",…,0.0,"""g""",100.0,"""g"""
"""0002c6e437ae0c4f3a9129ba13a702…",2020-11-27,"""kilocalories""","""255""",…,255.0,"""kcal""",100.0,"""g"""
"""0002c6e437ae0c4f3a9129ba13a702…",2020-11-27,"""proteins""","""34""",…,34.0,"""g""",100.0,"""g"""
…,…,…,…,…,…,…,…,…
"""0002c6e437ae0c4f3a9129ba13a702…",2020-11-28,"""carbohydrates""","""0""",…,0.0,"""g""",100.0,"""g"""
"""0002c6e437ae0c4f3a9129ba13a702…",2020-11-28,"""energy""","""1066.92""",…,1066.92,"""kJ""",100.0,"""g"""


In [35]:
# Show representative stored before/after values for each incompatible pair and three valid measures.
with pl.Config(tbl_rows=12, tbl_cols=8):
    display(
        duck.sql("""
        SELECT nutrient_name_raw, nutrient_value_raw, nutrient_unit_raw,
            nutrient_name, nutrient_value, nutrient_unit,
            unit_conversion_factor, unit_standardization_status
        FROM nutritionals
        WHERE dq_incompatible_nutrient_unit OR nutrient_name IN ('proteins', 'energy', 'kilocalories')
        QUALIFY row_number() OVER (
            PARTITION BY nutrient_name_raw, nutrient_unit_raw
            ORDER BY daltix_id, shop, country, download_date
        ) = 1
        ORDER BY nutrient_name_raw, nutrient_unit_raw
    """).pl()
    )

nutrient_name_raw,nutrient_value_raw,nutrient_unit_raw,nutrient_name,nutrient_value,nutrient_unit,unit_conversion_factor,unit_standardization_status
str,str,str,str,f64,str,f64,str
"""energy""","""1032""","""g""","""energy""",null,"""kJ""",null,"""incompatible_unit"""
"""energy""","""1066.92""","""kJ""","""energy""",1066.92,"""kJ""",1.0,"""identity"""
"""fats""","""962.32""","""kJ""","""fats""",null,"""g""",null,"""incompatible_unit"""
"""kilocalories""","""255""","""kcal""","""kilocalories""",255.0,"""kcal""",1.0,"""identity"""
"""polyunsaturated_fats""","""1493.688""","""kJ""","""polyunsaturated_fats""",null,"""g""",null,"""incompatible_unit"""
"""proteins""","""34""","""g""","""proteins""",34.0,"""g""",1.0,"""identity"""
"""salt""","""100""","""ml""","""salt""",null,"""g""",null,"""incompatible_unit"""
"""sugars""","""7570.82""","""ml""","""sugars""",null,"""g""",null,"""incompatible_unit"""
"""unsaturated_fats""","""1493.688""","""kJ""","""unsaturated_fats""",null,"""g""",null,"""incompatible_unit"""


In [36]:
# Retain an exact check of the published long grain.
duck.sql(
    "SELECT count(*) AS rows, count(DISTINCT (daltix_id, shop, country, download_date, nutrient_name)) AS nutrient_grains FROM nutritionals"
).pl()

rows,nutrient_grains
i64,i64
9467978,9467978


In [37]:
# Monitor canonical consistency separately from exceptions in the original units.
display(show_metrics("nutritionals"))
with pl.Config(tbl_rows=20, tbl_cols=6):
    display(
        duck.sql("""
        SELECT nutrient_name, min(nutrient_unit) AS canonical_unit,
            count(DISTINCT nutrient_unit) AS distinct_units, count(*) AS rows,
            count(*) FILTER (WHERE nutrient_value IS NULL) AS unknown_values
        FROM nutritionals GROUP BY nutrient_name ORDER BY nutrient_name
    """).pl()
    )
duck.sql("""
    SELECT nutrient_name_raw, nutrient_unit_raw, nutrient_unit AS canonical_unit, count(*) AS rows
    FROM nutritionals WHERE dq_incompatible_nutrient_unit
    GROUP BY ALL ORDER BY rows DESC, nutrient_name_raw
""").pl()

metric,value
str,str
"""rows""","""9467978"""
"""distinct_products""","""54155"""
"""distinct_nutrients""","""14"""
"""missing_numeric_values""","""70"""
"""non_numeric_values""","""0"""
"""missing_units""","""0"""
…,…
"""exact_duplicates_removed""","""0"""
"""canonical_observations""","""1096442"""


nutrient_name,canonical_unit,distinct_units,rows,unknown_values
str,str,i64,i64,i64
"""carbohydrates""","""g""",1,1085904,0
"""energy""","""kJ""",1,1081902,30
"""fats""","""g""",1,1080287,27
"""fibers""","""g""",1,764048,0
"""kilocalories""","""kcal""",1,1081902,0
"""monounsaturated_fats""","""g""",1,9213,0
"""polyols""","""g""",1,20998,0
"""polyunsaturated_fats""","""g""",1,9343,1
"""proteins""","""g""",1,1083536,0


nutrient_name_raw,nutrient_unit_raw,canonical_unit,rows
str,str,str,i64
"""energy""","""g""","""kJ""",30
"""fats""","""kJ""","""g""",27
"""sugars""","""ml""","""g""",8
"""salt""","""ml""","""g""",3
"""polyunsaturated_fats""","""kJ""","""g""",1
"""unsaturated_fats""","""kJ""","""g""",1


In [38]:
# Flag mass exceeding a mass portion without deleting values or assuming a density for liquids.
duck.sql("""
    SELECT nutrient_name, count(*) AS flagged_rows, count(DISTINCT daltix_id) AS products,
        min(nutrient_value) AS min_value, max(nutrient_value) AS max_value
    FROM nutritionals WHERE dq_mass_exceeds_portion
    GROUP BY nutrient_name ORDER BY nutrient_name
""").pl()

nutrient_name,flagged_rows,products,min_value,max_value
str,i64,i64,f64,f64
"""carbohydrates""",55,5,114.285714,986.666667
"""fats""",52,6,110.0,626.666667
"""fibers""",42,1,180.0,180.0
"""proteins""",73,3,206.666667,525.0
"""saturated_fats""",1,1,113.333333,113.333333
"""sugars""",2,2,193.5,303.333333


In [39]:
# Confirm that each selected observation retains its original portion basis.
display(
    duck.sql(
        "SELECT portion_value, portion_unit, count(DISTINCT (daltix_id, shop, country, download_date)) AS observations FROM nutritionals GROUP BY portion_value, portion_unit"
    ).pl()
)
checkpoint_result("nutritionals")

portion_value,portion_unit,observations
f64,str,i64
100.0,"""g""",1034855
100.0,"""ml""",61587


{'dataset': 'nutritionals',
 'persisted_rows': 9467978,
 'contracts': 'passed before publication'}

### 5.1 Conclusion

**Established:** 9,467,978 rows, 54,155 products and 14 nutrients, each with one canonical unit. All source values and selected versions remain traceable.

**Caveats:** 70 incompatible-unit rows have canonical `NULL`, with originals preserved; 225 rows exceed the 100 g mass basis and remain flagged. Of 100 version conflicts, 91 need the deterministic hash tie-break, which does not establish truth.

**Silver decision:** use `g` for mass, `kJ` for `energy` and `kcal` for `kilocalories`. No valid values needed scaling. Gold can reference these units in `dim_nutrient`, but its fact must retain the 100 g / 100 ml basis and DQ context. Download date is not an assumed effective-from date. [Full validation](../docs/silver-data-quality.md).


## 6. Weekly Products

**Purpose / grain:** one weekly product reference per source `daltix_id`.

**Issues → treatment:** normalize missing tokens; use `products` only where shop/country/language agree. Fill missing attributes only; existing primary values always win and literal disagreements remain flagged.

**Contract:** unique IDs, preserved rows, complete context and no context-unsafe filling. Zero enrichment is a valid result. [Implementation](../src/retail_pricing/silver_products.py).


In [40]:
# Inspect the primary source and the fallback used by the official run.
weekly_products_raw = pl.read_parquet(RAW_DIR / "weekly_prices_products.parquet")
display(weekly_products_raw.head())
products.head()

daltix_id,shop,name,brand,country,description,language,categories
str,str,str,str,str,str,str,str
"""9826e957eda31fb5a7a5a3ce5f2e9c…","""lld""","""prosecco | brut | bio""","""anna perenna""","""be""","""""","""nl""","""[ [ ""Bio, Eco en Fairtra…"
"""33e67943cd5d640537699ae4303d94…","""idla""","""enrico mori® boxershorts of sl…","""enrico mori®""","""be""","""viscose/polyamide/elastaan, na…","""nl""",null
"""79995654e06c3cd0d1827d71ec95dc…","""lld""","""tandpasta | total | herstelt d…","""colgate""","""be""","""colgate tandpasta total dageli…","""nl""","""[ [ ""Hygiëne en verzorgi…"
"""7059cf765f740c27b0808e9b8bd6b2…","""idla""","""polshorloge met hartslagmeter""",null,"""be""","""draadloze overdracht van de ha…","""nl""","""[ [ ""Onze aanbiedingen"",…"
"""eb199ff50711ac7b3e98e18dca48e3…","""idla""","""biospreads""","""palazzo bio®""","""be""","""lekker als broodbeleg of om te…","""nl""",null


daltix_id,shop,name,brand,…,description,categories,language,shop_name
str,str,str,str,…,str,str,str,str
"""a3ed88636b0d973d4cd419c19a0116…","""aldi""","""PALAZZO® Sauce pâtes rouge""","""palazzo®""",…,"""choix entre sauce légumes gril…","""[ [ ""Produits"", ""Ass…","""fr""","""Aldi"""
"""c38129dc388664ce48e060eebb9598…","""crf""","""Axe BodySpray Deodorant You 15…","""axe""",…,null,"""[ [ ""Verzorging & hygiën…","""nl""","""Carrefour"""
"""15fbf0a66ff12e20fde41913aebd72…","""aldi""","""ALL SEASONS® Fijnproeverssla""","""all seasons®""",…,"""gebruiksklare mengeling van kr…","""[ [ ""Producten"", ""As…","""nl""","""Aldi"""
"""7c93bb76b2772b001b264b779cb60c…","""ah""","""Blistex Lip relief crème""","""blistex""",…,"""lip relief cream helpt bij pij…","""[ [ ""Baby, verzorging en…","""nl""","""Albert Heijn"""
"""d60208deb02f76de96a8f771b1652d…","""crf""","""Cartec Achteruitkijkspiegel me…","""cartec""",…,null,"""[ [ ""Niet-voeding"" ], …","""nl""","""Carrefour"""


In [41]:
# Read the official weekly reference rather than implementing enrichment again.
weekly_products = pl.read_parquet(SILVER_DIR / SILVER_FILES["weekly_products"])
weekly_products.head()

daltix_id,shop,name,brand,…,dq_description_conflict,is_categories_enriched,dq_categories_conflict,shop_name
str,str,str,str,…,bool,bool,bool,str
"""9826e957eda31fb5a7a5a3ce5f2e9c…","""dll""","""prosecco | brut | bio""","""anna perenna""",…,false,false,true,"""Delhaize"""
"""33e67943cd5d640537699ae4303d94…","""aldi""","""enrico mori® boxershorts of sl…","""enrico mori®""",…,false,false,false,"""Aldi"""
"""79995654e06c3cd0d1827d71ec95dc…","""dll""","""tandpasta | total | herstelt d…","""colgate""",…,false,false,false,"""Delhaize"""
"""7059cf765f740c27b0808e9b8bd6b2…","""aldi""","""polshorloge met hartslagmeter""",null,…,false,false,false,"""Aldi"""
"""eb199ff50711ac7b3e98e18dca48e3…","""aldi""","""biospreads""","""palazzo bio®""",…,false,false,false,"""Aldi"""


In [42]:
# Show exact primary and fallback key cardinality before interpreting enrichment coverage.
pl.DataFrame(
    [
        {
            "source": "weekly_products",
            "rows": weekly_products.height,
            "keys": weekly_products["daltix_id"].n_unique(),
        },
        {
            "source": "products",
            "rows": products.height,
            "keys": products["daltix_id"].n_unique(),
        },
    ]
)

source,rows,keys
str,i64,i64
"""weekly_products""",114517,114517
"""products""",32826,32826


In [43]:
# Retain the missingness baseline and the tokens actually normalized by the official output.
display(missingness(weekly_products_raw, weekly_products, attributes))
converted_tokens(weekly_products_raw, weekly_products, attributes)

column,raw_nulls,silver_nulls
str,i64,i64
"""name""",0,170
"""brand""",6610,10325
"""description""",0,19334
"""categories""",23294,23294


column,raw_value,len
str,str,u32
"""brand""","""""",2979
"""brand""","""*""",69
"""brand""",""".""",23
"""brand""","""geen""",1
"""brand""","""no brand""",235
"""brand""","""not provided""",381
…,…,…
"""description""","""n/a""",33
"""description""","""x""",4104


In [44]:
# Report fallback overlap and contextual safety from the official join.
show_metrics(
    "weekly_products",
    ["fallback_matches", "safe_fallback_matches", "context_mismatches"],
)

metric,value
str,str
"""fallback_matches""","""9406"""
"""safe_fallback_matches""","""9406"""
"""context_mismatches""","""0"""


In [45]:
# Measure recovered attributes, remaining gaps and literal source disagreements.
show_metrics("weekly_products")

metric,value
str,str
"""rows""","""114517"""
"""fallback_matches""","""9406"""
"""safe_fallback_matches""","""9406"""
"""name_literal_differences""","""9356"""
"""name_case_only_differences""","""7651"""
"""brand_literal_differences""","""102"""
…,…
"""brands_enriched""","""0"""
"""descriptions_enriched""","""0"""


In [46]:
# Inspect the final reference without temporary fallback columns.
weekly_products.head()

daltix_id,shop,name,brand,…,dq_description_conflict,is_categories_enriched,dq_categories_conflict,shop_name
str,str,str,str,…,bool,bool,bool,str
"""9826e957eda31fb5a7a5a3ce5f2e9c…","""dll""","""prosecco | brut | bio""","""anna perenna""",…,false,false,true,"""Delhaize"""
"""33e67943cd5d640537699ae4303d94…","""aldi""","""enrico mori® boxershorts of sl…","""enrico mori®""",…,false,false,false,"""Aldi"""
"""79995654e06c3cd0d1827d71ec95dc…","""dll""","""tandpasta | total | herstelt d…","""colgate""",…,false,false,false,"""Delhaize"""
"""7059cf765f740c27b0808e9b8bd6b2…","""aldi""","""polshorloge met hartslagmeter""",null,…,false,false,false,"""Aldi"""
"""eb199ff50711ac7b3e98e18dca48e3…","""aldi""","""biospreads""","""palazzo bio®""",…,false,false,false,"""Aldi"""


In [47]:
# Confirm that the published reference preserved its row count and unique ID.
checkpoint_result("weekly_products")

{'dataset': 'weekly_products',
 'persisted_rows': 114517,
 'contracts': 'passed before publication'}

In [48]:
# The conclusion is computed from the checkpoint metrics.
weekly = metrics["weekly_products"]
display(
    Markdown(
        "### 6.1 Conclusion\n\n"
        f"**Established:** {weekly['rows']:,} products; {weekly['safe_fallback_matches']:,} compatible "
        f"fallback matches, with {weekly['names_enriched'] + weekly['brands_enriched'] + weekly['descriptions_enriched'] + weekly['categories_enriched']:,} "
        "attributes filled.\n\n"
        f"**Caveats:** {weekly['remaining_missing_name']:,} names, {weekly['remaining_missing_brand']:,} brands, "
        f"{weekly['remaining_missing_description']:,} descriptions and {weekly['remaining_missing_categories']:,} "
        "categories remain missing (placeholders included). Literal conflicts include case-only differences.\n\n"
        "**Silver decision:** Retain primary values and safe fallback logic for future snapshots. Do not "
        "treat text disagreement alone as evidence that either source is wrong."
    )
)

### 6.1 Conclusion

**Established:** 114,517 products; 9,406 compatible fallback matches, with 0 attributes filled.

**Caveats:** 170 names, 10,325 brands, 19,334 descriptions and 23,294 categories remain missing (placeholders included). Literal conflicts include case-only differences.

**Silver decision:** Retain primary values and safe fallback logic for future snapshots. Do not treat text disagreement alone as evidence that either source is wrong.

### 6.2 Category Path Evidence

**Established:** 54,242 product paths and 100,518 weekly paths are preserved in one
source-scoped child. Original parent `categories` values and path order are retained.

**Method / model:** parse JSON into typed paths; apply Unicode NFC and edge trim;
compare exact path tuples to flag duplicates, prefixes and repeated labels. A
reviewed source-specific signature flags navigation contamination. Preserve source
context and the parent-value hash. **No ML model or translation is used.**

**Caveat / decision:** these are structural checks, not approval of a business
classification. No primary category is assigned here. Gold applies separate
membership eligibility; the proposed English taxonomy will use a reviewed,
versioned mapping alongside the original paths.

[Implemented rules and lineage](../docs/silver-data-quality.md#categories-and-the-english-fmcg-category) ·
[English taxonomy proposal and evaluated models](../docs/silver-data-quality.md#categories-and-the-english-fmcg-category).


In [49]:
# Inspect persisted category evidence without selecting or deriving an analytical hierarchy.
display(show_metrics("product_category_paths"))
duck.sql("""
    SELECT source_dataset, path_review_status, count(*) AS path_rows
    FROM product_category_paths GROUP BY ALL ORDER BY source_dataset, path_review_status
""").pl()

metric,value
str,str
"""rows""","""154760"""
"""products_products""","""32826"""
"""products_missing_categories""","""497"""
"""products_products_with_multipl…","""6926"""
"""products_paths""","""54242"""
"""products_products_with_multipl…","""2218"""
…,…
"""weekly_prices_products_product…","""2000"""
"""weekly_prices_products_repeate…","""3482"""


source_dataset,path_review_status,path_rows
str,str,i64
"""products""","""ancestor_path""",19251
"""products""","""multiple_branches_review_requi…",4875
"""products""","""repeated_label_review_required""",160
"""products""","""unmapped""",29956
"""weekly_prices_products""","""ancestor_path""",5344
"""weekly_prices_products""","""multiple_branches_review_requi…",4060
"""weekly_prices_products""","""navigation_review_required""",3183
"""weekly_prices_products""","""repeated_label_review_required""",299
"""weekly_prices_products""","""unmapped""",87632


## 7. Weekly Locations

**Purpose / grain:** one weekly location per `(shop, location)`.

**Issues → treatment:** decode the nine verified reversed `shop_type` labels in Silver, preserving `shop_type_raw` and an explicit reversal flag. Retain optional geography; exclude ambiguous fallback keys and check contradictions before filling missing attributes.

**Contract:** unique key, preserved rows and missingness, known readable shop types, unchanged Raw lineage, valid coordinate pairs and no contradictory enrichment. Unreviewed shop-type labels stop publication. [Implementation](../src/retail_pricing/silver_locations.py) · [Mapping](../src/retail_pricing/reference.py).


In [50]:
# Inspect the primary locations and the non-weekly fallback reference.
weekly_locations_raw = pl.read_parquet(RAW_DIR / "weekly_prices_locations.parquet")
display(weekly_locations_raw.head())
locations.head()

shop,location,location_name,shop_type,…,geolocation_longitude,locality,postcode,state
str,str,str,str,…,f64,str,str,str
"""idla""","""de""","""Default""",null,…,null,null,null,null
"""idla""","""binche""","""Binche""",null,…,4.168999,"""Binche""","""7130 BINCHE""","""Wallonie"""
"""idla""","""dottenijs2""","""Dottenijs Opleidingscentrum""",null,…,3.3048918,"""Mouscron""","""7711""","""Wallonie"""
"""idla""","""lodelinsart""","""Lodelinsart""",null,…,4.4341463,"""Charleroi""","""6040""","""Wallonie"""
"""idla""","""péruwelz""","""Péruwelz""",null,…,3.6004418,"""Péruwelz""","""7601""","""Wallonie"""


shop,country_code,id,geolocation_latitude,…,dq_invalid_longitude,dq_incomplete_coordinates,dq_outside_benelux,shop_name
str,str,str,f64,…,bool,bool,bool,str
"""lidl""","""nl""","""1a131""",null,…,false,false,false,"""Lidl"""
"""lidl""","""de""","""5f02f""",null,…,false,false,false,"""Lidl"""
"""cg""","""be""","""f02af""",51.015406,…,false,false,false,"""Collect&Go"""
"""crf""","""be""","""64003""",50.908945,…,false,false,false,"""Carrefour"""
"""crf""","""be""","""dd6cf""",50.492171,…,false,false,false,"""Carrefour"""


In [51]:
# Read the official output including context-safety flags.
weekly_locations = pl.read_parquet(SILVER_DIR / SILVER_FILES["weekly_locations"])
weekly_locations.head()

shop,location,location_name,shop_type,…,dq_outside_benelux,location_type,region,shop_name
str,str,str,str,…,bool,str,str,str
"""aldi""","""de""","""Default""",null,…,false,"""National price (online)""",null,"""Aldi"""
"""aldi""","""binche""","""Binche""",null,…,false,"""Store""",null,"""Aldi"""
"""aldi""","""dottenijs2""","""Dottenijs Opleidingscentrum""",null,…,false,"""Store""","""Wallonia""","""Aldi"""
"""aldi""","""lodelinsart""","""Lodelinsart""",null,…,false,"""Store""","""Wallonia""","""Aldi"""
"""aldi""","""péruwelz""","""Péruwelz""",null,…,false,"""Store""","""Wallonia""","""Aldi"""


In [52]:
# Retain the exact retailer/location key check.
weekly_locations.select(
    pl.len().alias("rows"),
    pl.struct("shop", "location").n_unique().alias("business_keys"),
)

rows,business_keys
u32,u32
1230,1230


In [53]:
# Inspect geographic validity on the final values after any accepted enrichment.
weekly_locations.select(
    "shop",
    "location",
    "dq_invalid_latitude",
    "dq_invalid_longitude",
    "dq_incomplete_coordinates",
).head()

shop,location,dq_invalid_latitude,dq_invalid_longitude,dq_incomplete_coordinates
str,str,bool,bool,bool
"""aldi""","""de""",false,false,false
"""aldi""","""binche""",false,false,false
"""aldi""","""dottenijs2""",false,false,false
"""aldi""","""lodelinsart""",false,false,false
"""aldi""","""péruwelz""",false,false,false


In [54]:
# Preserve missingness evidence for every optional location attribute.
missingness(
    weekly_locations_raw,
    weekly_locations,
    [
        "shop_type",
        "postcode",
        "locality",
        "state",
        "geolocation_latitude",
        "geolocation_longitude",
    ],
)

column,raw_nulls,silver_nulls
str,i64,i64
"""shop_type""",436,436
"""postcode""",23,23
"""locality""",10,10
"""state""",10,10
"""geolocation_latitude""",6,6
"""geolocation_longitude""",6,6


In [55]:
# Show decoded labels and retained Raw values without implementing normalization in the notebook.
weekly_locations.group_by(
    "shop_type_raw", "shop_type", "is_shop_type_reversed"
).len().sort("shop_type", nulls_last=True)

shop_type_raw,shop_type,is_shop_type_reversed,len
str,str,bool,u32
"""da""","""ad""",true,222
"""sserpxe""","""express""",true,2
"""reileta hserf""","""fresh atelier""",true,8
"""tkramrepyh""","""hypermarkt""",true,26
"""tekram""","""market""",true,13
"""hgadtsem""","""mestdagh""",true,5
"""yxorp""","""proxy""",true,245
"""og & pohs""","""shop & go""",true,146
"""tekramrepus""","""supermarket""",true,127


In [56]:
# Inspect the eligible fallback population without silently resolving ambiguous keys.
locations.filter(~pl.col("dq_business_key_collision")).select(
    "shop",
    "id",
    "country_code",
    "postcode",
    "geolocation_latitude",
    "geolocation_longitude",
).head()

shop,id,country_code,postcode,geolocation_latitude,geolocation_longitude
str,str,str,str,f64,f64
"""lidl""","""1a131""","""nl""",null,null,null
"""lidl""","""5f02f""","""de""",null,null,null
"""cg""","""f02af""","""be""","""9051""",51.015406,3.6528982
"""crf""","""64003""","""be""","""1780""",50.908945,4.3048594
"""crf""","""dd6cf""","""be""","""7010""",50.492171,3.9675296


In [57]:
# Report safe matches from the official enrichment result.
show_metrics(
    "weekly_locations", ["rows", "safe_fallback_matches", "context_mismatches"]
)

metric,value
str,str
"""rows""","""1230"""
"""safe_fallback_matches""","""0"""
"""context_mismatches""","""0"""


In [58]:
# Retain postcode and coordinate disagreement checks as visible monitoring evidence.
weekly_locations.select(
    [
        pl.col(c).sum().alias(c)
        for c in [
            "dq_postcode_conflict",
            "dq_latitude_conflict",
            "dq_longitude_conflict",
            "dq_enrichment_context_mismatch",
        ]
    ]
)

dq_postcode_conflict,dq_latitude_conflict,dq_longitude_conflict,dq_enrichment_context_mismatch
u32,u32,u32,u32
0,0,0,0


In [59]:
# Measure recovered geography and the gaps deliberately retained.
show_metrics("weekly_locations")

metric,value
str,str
"""rows""","""1230"""
"""safe_fallback_matches""","""0"""
"""context_mismatches""","""0"""
"""postcodes_enriched""","""0"""
"""coordinates_enriched""","""0"""
"""countries_enriched""","""0"""
…,…
"""missing_shop_type""","""436"""
"""outside_benelux""","""1"""


In [60]:
# Inspect the final reference without temporary fallback fields.
weekly_locations.head()

shop,location,location_name,shop_type,…,dq_outside_benelux,location_type,region,shop_name
str,str,str,str,…,bool,str,str,str
"""aldi""","""de""","""Default""",null,…,false,"""National price (online)""",null,"""Aldi"""
"""aldi""","""binche""","""Binche""",null,…,false,"""Store""",null,"""Aldi"""
"""aldi""","""dottenijs2""","""Dottenijs Opleidingscentrum""",null,…,false,"""Store""","""Wallonia""","""Aldi"""
"""aldi""","""lodelinsart""","""Lodelinsart""",null,…,false,"""Store""","""Wallonia""","""Aldi"""
"""aldi""","""péruwelz""","""Péruwelz""",null,…,false,"""Store""","""Wallonia""","""Aldi"""


In [61]:
# Confirm the persisted weekly location checkpoint.
checkpoint_result("weekly_locations")

{'dataset': 'weekly_locations',
 'persisted_rows': 1230,
 'contracts': 'passed before publication'}

### 7.1 Conclusion

**Established:** 1,230 unique shop/location pairs; 794 shop-type values decoded across nine labels, with original values preserved. Zero safe fallback matches and zero enriched attributes.

**Caveats:** 436 missing shop types, 23 missing postcodes and six missing coordinate pairs remain. Decoded source labels are not a universal retail taxonomy; `mestdagh` remains a historical operator label.

**Silver decision:** Gold consumes the readable `shop_type` directly. Use the shared canonical `shop` code; retain gaps and descriptive names. Never reverse arbitrary text or infer country from shop. [Audit and evidence](../docs/silver-data-quality.md#products-and-locations).

**Geographic limit:** valid latitude/longitude ranges do not establish a correct
store location. Filipstad is preserved from Raw, but the other location reference
labels its Swedish coordinates `be`. No coordinate or country was invented; see
the [geographic investigation](../docs/gold-model.md).


### 7.2 Price context type, region and geographic review flag

**Purpose:** the client report needs to tell a national base price from a store price,
and to group stores by region, without inventing geography. Three small rules in
[`reference.py`](../src/retail_pricing/reference.py) do that in Silver:

- `location_type`: a location named `Default` is a **National price (online)** context,
  every other one a **Store**. The source does not say "online"; the label follows the
  adopted reading that a base price applies to the retailer as a whole.
- `region`: **Brussels**, **Flanders** or **Wallonia** from a *four-digit* Belgian
  postcode (1000–1299, 1300–1499, 1500–3999, 4000–7999, 8000–9999). Any other shape
  (missing, Dutch, five digits, text) gets no region, never a guess.
- `dq_outside_benelux`: coordinates outside a Benelux bounding box are flagged for
  review and kept unchanged, in both location tables.

In [62]:
display(weekly_locations.group_by("location_type").len().sort("len", descending=True))
weekly_locations.group_by("region").len().sort("len", descending=True)

location_type,len
str,u32
"""Store""",1224
"""National price (online)""",6


region,len
str,u32
"""Flanders""",716
"""Wallonia""",376
"""Brussels""",111
null,27


In [63]:
# Flagged, not removed: the row and its coordinates are exactly as in Raw.
display(show_metrics("locations", ["rows", "outside_benelux"]))
weekly_locations.filter(pl.col("dq_outside_benelux")).select(
    "shop",
    "location",
    "postcode",
    "locality",
    "state",
    "geolocation_latitude",
    "geolocation_longitude",
)

metric,value
str,str
"""rows""","""1638"""
"""outside_benelux""","""1"""


shop,location,postcode,locality,state,geolocation_latitude,geolocation_longitude
str,str,str,str,str,f64,f64
"""aldi""","""filipstad""","""68234""","""Filipstad""","""Värmlands län""",59.713997,14.169844


In [64]:
# Cross-check the postcode rule with the state text supplied by the source.
source_region = {
    "Vlaanderen": "Flanders",
    "Wallonie": "Wallonia",
    "Région de Bruxelles-Capitale - Brussels Hoofdstedelijk Gewest": "Brussels",
}
check = weekly_locations.filter(pl.col("region").is_not_null()).with_columns(
    pl.col("state").replace_strict(source_region, default=None).alias("source_region")
)
display(check.group_by("region", "source_region").len().sort("region", "source_region"))
check.filter(
    pl.col("source_region").is_not_null()
    & (pl.col("source_region") != pl.col("region"))
).select("shop", "location", "postcode", "locality", "state", "region")

region,source_region,len
str,str,u32
"""Brussels""","""Brussels""",111
"""Flanders""",null,1
"""Flanders""","""Brussels""",1
"""Flanders""","""Flanders""",714
"""Wallonia""","""Wallonia""",376


shop,location,postcode,locality,state,region
str,str,str,str,str,str
"""crf""","""strombeek-bever""","""1853""","""Ville de Bruxelles - Stad Brus…","""Région de Bruxelles-Capitale -…","""Flanders"""


In [65]:
# The price contexts that carry prices: stores without a postcode have no region.
duck.sql(
    "SELECT l.shop, l.location, l.location_type, l.region, l.postcode, l.state "
    "FROM weekly_locations l JOIN "
    "(SELECT DISTINCT shop, location FROM weekly_prices) p USING (shop, location) "
    "ORDER BY l.shop, l.location"
).pl()

shop,location,location_type,region,postcode,state
str,str,str,str,str,str
"""aldi""","""base""","""National price (online)""",null,null,null
"""clp""","""anderlecht-veeweyde""","""Store""",null,null,"""Région de Bruxelles-Capitale -…"
"""clp""","""ans""","""Store""","""Wallonia""","""4430""","""Wallonie"""
"""clp""","""beveren""","""Store""",null,null,null
"""clp""","""halle""","""Store""","""Flanders""","""1500""","""Vlaanderen"""
"""crf""","""base""","""National price (online)""",null,null,null
…,…,…,…,…,…
"""crf""","""mont-st-jean""","""Store""","""Wallonia""","""1410""","""Wallonie"""
"""crf""","""sint-agatha-berchem""","""Store""","""Brussels""","""1083""","""Région de Bruxelles-Capitale -…"


In [66]:
priced = duck.sql(
    "SELECT l.location_type, l.region FROM weekly_locations l JOIN "
    "(SELECT DISTINCT shop, location FROM weekly_prices) p USING (shop, location)"
).pl()
stores = priced.filter(pl.col("location_type") == "Store")
display(
    Markdown(
        "### 7.2 Conclusion\n\n"
        f"**Established:** {priced.height} price contexts: "
        f"{priced.height - stores.height} national base prices and {stores.height} stores. "
        f"{stores.filter(pl.col('region').is_not_null()).height} of the stores get a region "
        "from their postcode.\n\n"
        f"**Caveats:** {stores.filter(pl.col('region').is_null()).height} stores have no "
        "postcode, so their region stays empty even where the source state text suggests one; "
        "that case is listed for review instead of being filled in.\n\n"
        "**Silver decision:** deterministic type and region from a small reference, plus a review "
        "flag for far-away coordinates."
    )
)

### 7.2 Conclusion

**Established:** 16 price contexts: 3 national base prices and 13 stores. 11 of the stores get a region from their postcode.

**Caveats:** 2 stores have no postcode, so their region stays empty even where the source state text suggests one; that case is listed for review instead of being filled in.

**Silver decision:** deterministic type and region from a small reference, plus a review flag for far-away coordinates.

## 8. Weekly Prices

**Purpose / grain:** observations within `(daltix_id, shop, location, week)`; this working grain can contain multiple price versions, without implying invalid data.

**Issues → treatment:** collapse only exact six-field duplicates. Preserve different price pairs, mark ambiguity and derive source-specific promotion flags. The observation hash identifies a retained record; it does not repair the business grain or act as a Gold surrogate key.

**Contract:** complete fields, finite positive prices, no exact duplicates and consistent ambiguity flags. DuckDB writes directly from Parquet. [Implementation](../src/retail_pricing/silver_prices.py).


In [67]:
# Inspect the source schema without materializing the 19.2M-row table in Python.
duck.sql("DESCRIBE raw_weekly_prices").pl()

column_name,column_type,null,key,default,extra
str,str,str,str,str,str
"""shop""","""VARCHAR""","""YES""",null,null,null
"""location""","""VARCHAR""","""YES""",null,null,null
"""daltix_id""","""VARCHAR""","""YES""",null,null,null
"""week""","""DATE""","""YES""",null,null,null
"""price""","""DOUBLE""","""YES""",null,null,null
"""price_promo""","""DOUBLE""","""YES""",null,null,null


In [68]:
# Retain deduplication, ambiguity and promotion evidence from the official run.
display(show_metrics("weekly_prices"))
duck.sql("""
    SELECT count(*) FILTER (WHERE dq_source_exact_duplicate) AS retained_rows_from_exact_duplicates,
        count(*) FILTER (WHERE has_multiple_price_versions) AS multiple_version_rows,
        count(*) FILTER (WHERE NOT has_multiple_price_versions) AS single_version_rows,
        count(*) FILTER (WHERE dq_non_positive_price) AS non_positive_prices,
        count(*) FILTER (WHERE dq_non_positive_promo_price) AS non_positive_promos
    FROM weekly_prices
""").pl()

metric,value
str,str
"""raw_rows""","""19218071"""
"""rows""","""19075099"""
"""exact_duplicates_removed""","""142972"""
"""multiple_price_version_rows""","""751824"""
"""multiple_price_version_grains""","""375912"""
"""promotion_rows""","""1636220"""
"""promo_above_price_rows""","""10916"""


retained_rows_from_exact_duplicates,multiple_version_rows,single_version_rows,non_positive_prices,non_positive_promos
i64,i64,i64,i64,i64
142972,751824,18323275,0,0


In [69]:
# Inspect examples of retained alternatives instead of selecting a supposed true weekly price.
duck.sql("""
    SELECT daltix_id, shop, location, week, price, price_promo,
        price_version_count, price_resolution_rule
    FROM weekly_prices WHERE has_multiple_price_versions LIMIT 10
""").pl()

daltix_id,shop,location,week,price,price_promo,price_version_count,price_resolution_rule
str,str,str,date,f64,f64,i64,str
"""0000ba625520cd774f3fc738e27d9d…","""clp""","""beveren""",2020-12-28,13.491,13.491,2,"""multiple_price_observations"""
"""0000ba625520cd774f3fc738e27d9d…","""clp""","""beveren""",2020-12-28,16.489,16.489,2,"""multiple_price_observations"""
"""0000ba625520cd774f3fc738e27d9d…","""clp""","""halle""",2019-12-30,13.491,13.491,2,"""multiple_price_observations"""
"""0000ba625520cd774f3fc738e27d9d…","""clp""","""halle""",2019-12-30,14.99,14.99,2,"""multiple_price_observations"""
"""00032e04908fe565cbb358f6f5de48…","""crf""","""base""",2019-05-27,2.565,2.565,2,"""multiple_price_observations"""
"""00032e04908fe565cbb358f6f5de48…","""crf""","""base""",2019-05-27,2.85,2.85,2,"""multiple_price_observations"""
"""00032e04908fe565cbb358f6f5de48…","""crf""","""base""",2019-12-30,2.89,2.89,2,"""multiple_price_observations"""
"""00032e04908fe565cbb358f6f5de48…","""crf""","""base""",2019-12-30,3.223,3.223,2,"""multiple_price_observations"""
"""00032e04908fe565cbb358f6f5de48…","""crf""","""base""",2020-12-28,2.85,2.85,2,"""multiple_price_observations"""


In [70]:
# Retain post-write evidence that exact duplicates are absent and the version flag agrees with the count.
duck.sql("""
    SELECT count(*) AS rows,
        count(*) - count(DISTINCT (daltix_id, shop, location, week, price, price_promo)) AS exact_duplicate_rows,
        count(*) FILTER (WHERE has_multiple_price_versions IS DISTINCT FROM (price_version_count > 1)) AS inconsistent_version_flags
    FROM weekly_prices
""").pl()

rows,exact_duplicate_rows,inconsistent_version_flags
i64,i64,i64
19075099,0,0


### 8.1 Conclusion

**Established:** 19,218,071 Raw rows become 19,075,099 after removing 142,972 exact duplicates.

**Caveats:** 751,824 rows in 375,912 weekly grains remain ambiguous; 10,916 promotional values exceed regular price.

**Silver decision:** Retain alternatives and extremes. Gold must explicitly handle ambiguity before price aggregation or comparison; Silver does not select min/max/first as truth.


## 9. Primary FMCG Classification

**Purpose / grain:** a category manager needs one English category that means the same thing across retailers, while each retailer keeps its own navigation. The lookup has one decision per `(daltix_id, shop, country)`; original categories and every source path stay untouched.

**Method (`fmcg_v3`), in the order it is applied:**

1. **Retailer category paths decide first.** Exact, versioned rules in [`fmcg_taxonomy.py`](../src/retail_pricing/fmcg_taxonomy.py) match normalised Dutch merchandise labels; other languages abstain. Promotions, age facets and site navigation are not products. Documented boundaries win inside one path (pet or baby use over ingredients, frozen over the department, alcohol-free beverages, eggs over the baking department). Several paths must **agree**: competing categories are never settled by a first path or a majority vote.
2. **The product name is the backup.** When the paths cannot decide (no path, unmapped labels, unusable paths) or they disagree, the words of the name can: `tandpasta`, `lego`, `pedigree`. Exactly one category must be named, and for disagreeing paths it must be one of the competing candidates. A name never overrides a single agreeing path (section 9.2).
3. **Otherwise the product abstains** with a recorded reason (`Other / Unclassified`). **No model, translation or fuzzy matching runs in the pipeline.**

**Contract / handoff:** the official pipeline publishes `silver_product_primary_categories` with the rule ids, path references, candidates and mapping version. Gold performs a checked many-to-one join onto Product; keys, retailer categories and facts are unchanged. [Taxonomy, boundaries and coverage](../docs/silver-data-quality.md#categories-and-the-english-fmcg-category).

In [71]:
# Call the production rules on a few representative paths; the notebook holds no rule.
examples = [
    (["Babyvoeding", "6 maanden"], "nl", "ah"),
    (["Diepvries", "Vis"], "nl", "ah"),
    (["Huishouden", "Diepvrieszakjes"], "nl", "ah"),
    (["Wijn, bier, sterke drank", "Alcoholvrij"], "nl", "ah"),
    (["2+1 op Snoep", "Chocolade"], "nl", "ah"),
    (["Koffie"], "fr", "ah"),
]
pl.DataFrame(
    [
        {
            "path": " > ".join(path),
            "language": language,
            "category": sorted(codes),
            "rules": sorted(rules),
            "outcome": reason,
        }
        for path, language, shop in examples
        for codes, rules, reason in [classify_path(path, language, shop)]
    ]
)

path,language,category,rules,outcome
str,str,list[str],list[str],str
"""Babyvoeding > 6 maanden""","""nl""","[""baby_care""]","[""nl_label_1867b4bab237""]","""matched"""
"""Diepvries > Vis""","""nl""","[""frozen""]","[""explicit_frozen_food"", ""nl_label_61c3aba9bac0"", ""nl_label_ddbbfd66eec4""]","""matched"""
"""Huishouden > Diepvrieszakjes""","""nl""","[""household_supplies""]","[""nl_label_f5b1c176b7cf""]","""matched"""
"""Wijn, bier, sterke drank > Alc…","""nl""","[""non_alcoholic_beverages""]","[""explicit_alcohol_free"", ""nl_label_2c0a43ba8acf"", ""nl_label_9f19eeb4cf2c""]","""matched"""
"""2+1 op Snoep > Chocolade""","""nl""",[],[],"""excluded"""
"""Koffie""","""fr""",[],[],"""unsupported_language"""


In [72]:
# Read the official decisions and show why some products stay unclassified.
primary_category_decisions = pl.read_parquet(
    SILVER_DIR / SILVER_FILES["product_primary_categories"]
)
display(show_metrics("product_primary_categories"))
primary_category_decisions.group_by(
    "primary_fmcg_category_en", "fmcg_classification_status"
).len().sort("primary_fmcg_category_en", "fmcg_classification_status")

metric,value
str,str
"""rows""","""167077"""
"""classified""","""97116"""
"""conflicting_evidence""","""2120"""
"""excluded_evidence""","""6792"""
"""missing_evidence""","""46237"""
"""no_supported_match""","""14760"""
"""review_required""","""52"""


primary_fmcg_category_en,fmcg_classification_status,len
str,str,u32
"""Alcoholic Beverages""","""classified""",10710
"""Baby Care""","""classified""",1850
"""Bakery""","""classified""",1446
"""Beauty & Cosmetics""","""classified""",2827
"""Breakfast & Cereals""","""classified""",799
"""Chilled & Prepared Foods""","""classified""",1752
…,…,…
"""Other / Unclassified""","""review_required""",52
"""Pantry & Cooking""","""classified""",9534


In [73]:
# Coverage differs by retailer because category evidence differs, not because of quality.
duck.sql(
    "SELECT shop_name, count(*) AS identities, "
    "count(*) FILTER (WHERE fmcg_classification_status = 'classified') AS classified, "
    "round(100.0 * count(*) FILTER (WHERE fmcg_classification_status = 'classified') / count(*), 1) "
    "AS classified_pct, "
    "count(*) FILTER (WHERE fmcg_classification_status = 'missing_evidence') AS no_category_path "
    "FROM product_primary_categories GROUP BY shop_name ORDER BY shop_name"
).pl()

shop_name,identities,classified,classified_pct,no_category_path
str,i64,i64,f64,i64
"""Albert Heijn""",23652,7941,33.6,13764
"""Aldi""",14069,2567,18.2,3850
"""Carrefour""",46759,32676,69.9,8641
"""Collect&Go""",9871,5206,52.7,4068
"""Colruyt Lowest Prices""",24451,17295,70.7,5717
"""Delhaize""",33595,27331,81.4,743
"""Jumbo""",14454,4076,28.2,9454
"""Lidl""",226,24,10.6,0


### 9.1 Testing a new rule before adopting it

Coverage grows by adding exact labels, but a label is only worth adding if it is
safe. The test is deliberately strict: classify every product **with and without** the
label, and adopt it only when it classifies more products and changes none that are
already classified. Department labels such as *Kruidenierswaren* fail because they
mix categories: a product filed under both a pantry and a household path would turn
from a correct decision into a conflict. This is how `fmcg_v2` was built.

In [74]:
from collections import defaultdict

from retail_pricing import fmcg_taxonomy as taxonomy
from retail_pricing.silver_products import IDENTITY, classify_product_paths

category_paths = pl.read_parquet(SILVER_DIR / SILVER_FILES["product_category_paths"])
paths_by_product = defaultdict(list)
products_with_label = defaultdict(set)
for row in category_paths.iter_rows(named=True):
    product = tuple(row[field] for field in IDENTITY)
    paths_by_product[product].append(row)
    for label in row["category_path_normalized"]:
        products_with_label[taxonomy.normalize_label(label)].add(product)


def effect_of_label(label, category_code):
    """Compare the decisions of the products that carry `label`, without and with the rule."""
    key = taxonomy.normalize_label(label)
    adopted = taxonomy.ANCHORS.pop(key, None)
    products = products_with_label[key]
    before = {p: classify_product_paths(paths_by_product[p]) for p in products}
    taxonomy.ANCHORS[key] = category_code
    after = {p: classify_product_paths(paths_by_product[p]) for p in products}
    if adopted is None:
        del taxonomy.ANCHORS[key]  # leave the production rules exactly as they were
    else:
        taxonomy.ANCHORS[key] = adopted
    gained = changed = 0
    for p in products:
        was, now = before[p], after[p]
        if was["fmcg_classification_status"] != "classified":
            gained += now["fmcg_classification_status"] == "classified"
        elif now["primary_fmcg_category_code"] != was["primary_fmcg_category_code"]:
            changed += 1
    return {
        "label": label,
        "proposed_category": category_code,
        "products_with_label": len(products),
        "newly_classified": gained,
        "classified_that_change": changed,
        "verdict": "adopt" if gained and not changed else "reject",
    }


pl.DataFrame(
    [
        effect_of_label("Honing", "pantry_cooking"),
        effect_of_label("Snoep en kauwgom", "confectionery_snacks"),
        effect_of_label("Kruidenierswaren", "pantry_cooking"),
        effect_of_label("Dranken", "non_alcoholic_beverages"),
    ]
)

label,proposed_category,products_with_label,newly_classified,classified_that_change,verdict
str,str,i64,i64,i64,str
"""Honing""","""pantry_cooking""",84,73,0,"""adopt"""
"""Snoep en kauwgom""","""confectionery_snacks""",47,47,0,"""adopt"""
"""Kruidenierswaren""","""pantry_cooking""",4859,914,1948,"""reject"""
"""Dranken""","""non_alcoholic_beverages""",6183,887,3456,"""reject"""


### 9.2 When the paths cannot decide: the product name

**Why.** Almost half of the products that stay `Other / Unclassified` have no usable category path at all: a retailer's own assortment loaded without categories, or labels no rule maps. Their names are not ambiguous: *Colgate tandpasta*, *LEGO Duplo* or *Pedigree brokken* say what the product is.

**Where the vocabulary comes from.** It is built once, offline, and committed as an explicit list in [`fmcg_name_rules.py`](../src/retail_pricing/fmcg_name_rules.py); the pipeline only reads it, so the Silver build stays deterministic.

- Start from the products that the **category paths already classified** (weak, "silver" labels: no manual labelling needed).
- Keep a word when at least **20** such products carry it and at least **98%** of them agree on one category (`mine_name_keywords`).
- Reject the few words that pass the statistics but are not evidence, such as retailer names, colours and generic adjectives (`STOP_TOKENS`).

This is the usual hybrid pattern for product classification: an explicit taxonomy you control, deterministic rules for the clear cases, and a measured, reviewable fallback instead of a black box ([GS1 Global Product Classification](https://www.gs1belu.org/en/global-product-classification-gpc), [product categorization guide](https://wisepim.com/guides/product-categorization)).

**How it is checked.** On products the vocabulary has not seen: (1) a random 50/50 split, mining on one half and testing on the other; (2) *leave one shop out*, mining on the other retailers and testing on the held-out one, which is the case that matters for a retailer with poor paths. *Agreement* is the share of firing predictions that equal the path decision. It is an estimate, because the paths are not perfect either.

In [75]:
# Prepare the products whose category the paths decided, with their Dutch names.
from retail_pricing.fmcg_name_rules import (
    NAME_KEYWORDS_NL,
    mine_name_keywords,
    name_tokens,
)
from retail_pricing.fmcg_taxonomy import CATEGORY_LABELS

decisions = pl.read_parquet(SILVER_DIR / SILVER_FILES["product_primary_categories"])
product_names = pl.concat(
    [
        pl.read_parquet(SILVER_DIR / SILVER_FILES[name]).select(
            "daltix_id", "shop", "country", "language", "name"
        )
        for name in ("weekly_products", "products")  # the weekly value wins
    ]
).unique(subset=["daltix_id", "shop", "country"], keep="first", maintain_order=True)

decided_by_path = (
    decisions.filter(
        (pl.col("fmcg_classification_status") == "classified")
        & ~pl.col("evidence_source_datasets").list.contains("product_name")
    )
    .join(product_names, on=["daltix_id", "shop", "country"])
    .filter(pl.col("name").is_not_null() & (pl.col("language") == "nl"))
    .select("shop", "name", code="primary_fmcg_category_code")
)


def predict(vocabulary, name):
    """The single category the words of a name point to, otherwise None."""
    codes = {vocabulary[word][0] for word in name_tokens(name) if word in vocabulary}
    return next(iter(codes)) if len(codes) == 1 else None


def agreement(train, test):
    """Mine on `train`; compare the predictions with the path decision on `test`."""
    vocabulary = mine_name_keywords(zip(train["name"], train["code"]))
    fired = [
        (predict(vocabulary, name), code)
        for name, code in zip(test["name"], test["code"])
        if predict(vocabulary, name)
    ]
    return {
        "words_kept": len(vocabulary),
        "tested_products": test.height,
        "predictions": len(fired),
        "share_with_a_prediction": round(len(fired) / test.height, 3),
        "agreement_with_paths": round(sum(a == b for a, b in fired) / len(fired), 4),
    }


shuffled = decided_by_path.sample(fraction=1.0, shuffle=True, seed=11)
half = shuffled.height // 2
pl.DataFrame(
    [{"check": "random 50/50 split", **agreement(shuffled[:half], shuffled[half:])}]
)

check,words_kept,tested_products,predictions,share_with_a_prediction,agreement_with_paths
str,i64,i64,i64,f64,f64
"""random 50/50 split""",256,43765,10295,0.235,0.9873


In [76]:
# Leave one shop out: mine on the other retailers, test on the held-out one.
pl.DataFrame(
    [
        {
            "held_out_shop": shop,
            **agreement(
                decided_by_path.filter(pl.col("shop") != shop),
                decided_by_path.filter(pl.col("shop") == shop),
            ),
        }
        for shop in sorted(decided_by_path["shop"].unique())
        if decided_by_path.filter(pl.col("shop") == shop).height >= 100
    ]
)

held_out_shop,words_kept,tested_products,predictions,share_with_a_prediction,agreement_with_paths
str,i64,i64,i64,f64,f64
"""ah""",475,7674,1420,0.185,0.9704
"""aldi""",505,1435,208,0.145,0.9183
"""cg""",473,5089,1339,0.263,0.9888
"""clp""",439,14636,3482,0.238,0.9779
"""crf""",304,27912,7039,0.252,0.929
"""dll""",454,26897,4551,0.169,0.9695
"""jumbo""",482,3880,755,0.195,0.9788


On a random split the words fire on about a quarter of the products with 98.7% agreement; agreement stays between 92% and 99% even for a retailer the vocabulary never saw (Aldi, the smallest test, is the lowest). Most of the disagreement at Carrefour is the boundary between *Personal Care* and *Beauty & Cosmetics*, where retailers file the same brand differently; that is a taxonomy boundary, not an obviously wrong answer. The committed vocabulary is the one mined on **all** path-decided products:

In [77]:
# The committed vocabulary: words per category, and a few examples.
pl.DataFrame(
    [
        {
            "category": CATEGORY_LABELS[code],
            "words": len(words),
            "examples": ", ".join(words[:8]),
        }
        for code, words in NAME_KEYWORDS_NL.items()
    ]
).sort("words", descending=True)

category,words,examples
str,i64,str
"""Non-FMCG / General Merchandise""",132,"""adapter, artline, atoma, autol…"
"""Alcoholic Beverages""",61,"""bacardi, bacardí, beaujolais, …"
"""Household Care""",57,"""afwasmiddel, ajax, allesreinig…"
"""Personal Care & Hygiene""",53,"""absodys, aussie, bodyspray, ca…"
"""Pantry & Cooking""",36,"""barilla, basmati, bouillon, ca…"
"""Dairy & Eggs""",30,"""actimel, activia, arla, babybe…"
…,…,…
"""Household Paper & Supplies""",5,"""diepvrieszakken, huisvuilzakke…"
"""Bakery""",3,"""jacquet, melkbroodjes, toastbr…"


### 9.3 What the name changed, and what stays open

The table shows who decides today. **Category path** is the original method; **name only** are products without a usable path that a name settled; **path + name (tie-break)** are products whose paths disagreed and the name chose one of the competing candidates.

In [78]:
# Basis of every classified decision, per retailer.
basis = (
    decisions.filter(pl.col("fmcg_classification_status") == "classified")
    .with_columns(
        basis=pl.when(
            pl.col("matched_rule_ids").list.contains("name_breaks_path_conflict")
        )
        .then(pl.lit("path + name (tie-break)"))
        .when(pl.col("evidence_source_datasets").list.contains("product_name"))
        .then(pl.lit("name only"))
        .otherwise(pl.lit("category path"))
    )
    .group_by("shop_name", "basis")
    .len()
    .pivot(on="basis", index="shop_name", values="len")
    .fill_null(0)
    .sort("shop_name")
)
basis

shop_name,category path,path + name (tie-break),name only
str,u32,u32,u32
"""Albert Heijn""",7674,21,246
"""Aldi""",1435,4,1128
"""Carrefour""",27918,58,4700
"""Collect&Go""",5089,1,116
"""Colruyt Lowest Prices""",14636,8,2651
"""Delhaize""",26897,39,395
"""Jumbo""",3880,0,196
"""Lidl""",6,0,18


In [79]:
# A random sample of what the name alone decided, to judge the answers by eye.
(
    decisions.filter(pl.col("evidence_source_datasets").list.contains("product_name"))
    .filter(pl.col("evidence_path_references").list.len() == 0)
    .join(product_names, on=["daltix_id", "shop", "country"])
    .select("shop_name", "primary_fmcg_category_en", "name", "matched_rule_ids")
    .sample(n=14, seed=3)
)

shop_name,primary_fmcg_category_en,name,matched_rule_ids
str,str,str,list[str]
"""Colruyt Lowest Prices""","""Non-FMCG / General Merchandise""","""kangourou pennenbak cactus""","[""name:kangourou""]"
"""Carrefour""","""Non-FMCG / General Merchandise""","""sandisk geheugenkaart micro sd…","[""name:geheugenkaart"", ""name:sandisk"", ""name:sdhc""]"
"""Colruyt Lowest Prices""","""Non-FMCG / General Merchandise""","""sistema hydr. fles + rietje tr…","[""name:sistema""]"
"""Carrefour""","""Pantry & Cooking""","""soubry spirelli haver 375 g""","[""name:soubry""]"
"""Colruyt Lowest Prices""","""Non-FMCG / General Merchandise""","""ringmap pastelini paars a4""","[""name:ringmap""]"
"""Carrefour""","""Non-FMCG / General Merchandise""","""exacompta 100 stekkaarten 75x1…","[""name:exacompta""]"
…,…,…,…
"""Carrefour""","""Dairy & Eggs""","""la laitière rijstpap vanille 5…","[""name:laitière""]"
"""Carrefour""","""Coffee & Tea""","""jacqmotte creations espresso d…","[""name:jacqmotte""]"


**Compared with `fmcg_v2`.** v3 adds three path rules (*Verse eieren*, *Plantaardig alternatief*, *Tandpasta*) and the name evidence. Of the products v2 had classified, 37 plant-based drinks that Colruyt files under dairy now follow Carrefour, Delhaize and Albert Heijn into *Plant-Based Alternatives*, and one Aldi egg product that was filed under *Bakery* now abstains; nothing else changed. Aldi's diapers (filed under *Papierproducten*) remain the known boundary case.

What is still open is shown below: the labels that would classify the most remaining products if a rule existed. They are **not** adopted automatically. A bare label such as *Frankrijk* or *Online exclusiviteiten* looks 100% pure in this snapshot only because of how one retailer files its wines; without its parent path it means nothing, so a person decides.

In [80]:
# Unmapped path labels that co-occur with one category, for a person to review.
from retail_pricing.fmcg_taxonomy import ANCHORS, normalize_label
from retail_pricing.silver_products import IDENTITY

category_paths = pl.read_parquet(SILVER_DIR / SILVER_FILES["product_category_paths"])
path_decided = decisions.filter(
    (pl.col("fmcg_classification_status") == "classified")
    & ~pl.col("evidence_source_datasets").list.contains("product_name")
).select(*IDENTITY, code="primary_fmcg_category_code")
unmapped_labels = (
    category_paths.filter(
        ~pl.col("dq_duplicate_category_path")
        & ~pl.col("is_prefix_of_another_path")
        & ~pl.col("dq_navigation_contamination")
        & ~pl.col("dq_repeated_category_label")
        & (pl.col("language") == "nl")
    )
    .select(*IDENTITY, label="category_path_normalized")
    .explode("label", empty_as_null=True, keep_nulls=True)
    .unique()
    .with_columns(
        key=pl.col("label").map_elements(normalize_label, return_dtype=pl.String)
    )
    .filter(~pl.col("key").is_in(list(ANCHORS)))
    .join(path_decided, on=IDENTITY)
)
(
    unmapped_labels.group_by("key")
    .agg(
        products=pl.len(),
        proposed_category=pl.col("code").mode().first(),
        purity=(pl.col("code") == pl.col("code").mode().first()).mean().round(3),
    )
    .filter((pl.col("products") >= 30) & (pl.col("purity") >= 0.99))
    .sort("products", descending=True)
    .head(12)
)

key,products,proposed_category,purity
str,u32,str,f64
"""frankrijk""",1895,"""alcoholic_beverages""",0.998
"""vlees, vis en vegetarische pro…",1827,"""meat_poultry_seafood""",1.0
"""lichaam""",1258,"""personal_care""",1.0
"""kruidenierswaren/droge voeding""",1045,"""pantry_cooking""",0.996
"""haar""",875,"""personal_care""",1.0
"""koeken/chocolade/snoep""",862,"""confectionery_snacks""",1.0
"""huislinnen""",731,"""non_fmcg""",1.0
"""bordeaux""",719,"""alcoholic_beverages""",1.0
"""snoep, koek, chips en chocolad…",620,"""confectionery_snacks""",1.0


In [81]:
# The conclusion is computed from the published decisions.
decided = metrics["product_primary_categories"]
by_name = decisions.filter(
    pl.col("evidence_source_datasets").list.contains("product_name")
).height
display(
    Markdown(
        "### 9.4 Conclusion\n\n"
        f"**Established:** {decided['rows']:,} product identities have one decision each; "
        f"{decided['classified']:,} ({decided['classified'] / decided['rows']:.1%}) are "
        f"rule-supported, {by_name:,} of them with the product name as evidence. "
        "`fmcg_v2` classified 87,470 (52.4%).\n\n"
        f"**Caveats:** {decided['missing_evidence']:,} have neither a category path nor a name "
        f"a rule understands, {decided['no_supported_match']:,} have paths no rule supports, "
        f"{decided['excluded_evidence']:,} have only unusable paths, {decided['conflicting_evidence']:,} "
        f"conflict and {decided['review_required']} are withheld after a reviewed source contradiction. "
        "Coverage is not accuracy: agreement with the paths is an estimate, not a verification.\n\n"
        "**Silver decision:** abstain with a recorded reason instead of guessing. Gold exposes this "
        "category on Product and keeps the retailer categories as overlapping evidence."
    )
)

### 9.4 Conclusion

**Established:** 167,077 product identities have one decision each; 97,116 (58.1%) are rule-supported, 9,581 of them with the product name as evidence. `fmcg_v2` classified 87,470 (52.4%).

**Caveats:** 46,237 have neither a category path nor a name a rule understands, 14,760 have paths no rule supports, 6,792 have only unusable paths, 2,120 conflict and 52 are withheld after a reviewed source contradiction. Coverage is not accuracy: agreement with the paths is an estimate, not a verification.

**Silver decision:** abstain with a recorded reason instead of guessing. Gold exposes this category on Product and keeps the retailer categories as overlapping evidence.

## 10. Cross-Table Contracts

**Purpose:** establish safe cardinality and context on the published Silver tables. Full weekly location coverage is required; incomplete product or nutritional overlap is reported, not hidden.

These are structural checks, not permission to treat later reference attributes as historically effective. [Official checks](../src/retail_pricing/quality.py).


In [82]:
# Confirm all nine outputs belong to this completed checkpoint.
pl.DataFrame([checkpoint_result(name) for name in OUTPUTS])

dataset,persisted_rows,contracts
str,i64,str
"""products""",32826,"""passed before publication"""
"""locations""",1638,"""passed before publication"""
"""prices""",1198547,"""passed before publication"""
"""nutritionals""",9467978,"""passed before publication"""
"""weekly_products""",114517,"""passed before publication"""
"""weekly_locations""",1230,"""passed before publication"""
"""weekly_prices""",19075099,"""passed before publication"""
"""product_category_paths""",154760,"""passed before publication"""
"""product_primary_categories""",167077,"""passed before publication"""


In [83]:
# Retain weekly product coverage, row-preservation and context evidence.
show_metrics(
    "cross_table",
    [
        "weekly_price_rows",
        "weekly_product_joined_rows",
        "weekly_product_matched_rows",
        "weekly_product_unmatched_rows",
        "weekly_product_coverage_pct",
        "weekly_product_context_mismatches",
    ],
)

metric,value
str,str
"""weekly_price_rows""","""19075099"""
"""weekly_product_joined_rows""","""19075099"""
"""weekly_product_matched_rows""","""18323389"""
"""weekly_product_unmatched_rows""","""751710"""
"""weekly_product_coverage_pct""","""96.06"""
"""weekly_product_context_mismatc…","""0"""


In [84]:
# Retain the complete weekly location relationship and its cardinality evidence.
show_metrics(
    "cross_table",
    [
        "weekly_price_rows",
        "weekly_location_joined_rows",
        "weekly_location_unmatched_rows",
    ],
)

metric,value
str,str
"""weekly_price_rows""","""19075099"""
"""weekly_location_joined_rows""","""19075099"""
"""weekly_location_unmatched_rows""","""0"""


In [85]:
# Retain nutritional overlap and context evidence without joining temporal nutrient history to facts.
show_metrics(
    "cross_table",
    ["matched_nutritional_products", "nutrition_product_context_mismatches"],
)

metric,value
str,str
"""matched_nutritional_products""","""15696"""
"""nutrition_product_context_mism…","""0"""


### 10.1 Conclusion

**Established:** all table and cross-table contracts passed. Weekly product coverage is 96.06%; weekly location coverage is complete. Nutritionals overlap with 15,696 weekly products.

**Caveats:** 751,710 weekly price rows lack product metadata. Context agreement and safe cardinality do not establish temporal compatibility.

**Silver decision:** keep unmatched observations and separate sources. Define dimensional grain, unknown-member policy and historical enrichment rules explicitly in Gold.


## 11. Checkpoint and Gold Handoff

The complete pipeline runs independently of this notebook. The persisted manifest records Raw hashes, the source-code fingerprint, dependency versions, output schemas and hashes, metrics and timing.

**Handoff.** [Notebook 03](03_analytical_modelling.ipynb) explains the Gold model built from these nine datasets and [the pipeline](../src/retail_pricing/pipeline.py) publishes it. Silver keeps every market; Gold publishes the Belgian mart. Weekly price variation, price identities without a product (they become inferred products in Gold), the nutritional basis and the temporal limits all stay explicit.

**What Silver decided about categories.** The raw category JSON is preserved and the evidence table keeps every observed path, flagged for duplicates, prefixes, repeated labels and site navigation. One FMCG category per product is decided here (section 9) with an explicit reason whenever it abstains; retailer paths stay overlapping evidence for Gold. See the [category notes](../docs/silver-data-quality.md#categories-and-the-english-fmcg-category).

**Quality flags.** Weekly `has_multiple_price_versions` describes alternative price pairs without asserting invalidity. Nutrition checks retain negative values with flags and compare kJ and kcal only within the same selected payload and portion. Field lineage and limitations are in the [limits](../docs/silver-data-quality.md#limits-to-remember).


In [86]:
# Show the final persisted artifact inventory rather than constructing an unpersisted manifest.
pl.DataFrame(
    [
        {
            "dataset": f["dataset"],
            "rows": f["rows"],
            "file": f["file"],
            "size_mb": round(f["size_bytes"] / 1024**2, 2),
        }
        for f in manifest["files"]
    ]
)

dataset,rows,file,size_mb
str,i64,str,f64
"""products""",32826,"""silver_products.parquet""",3.51
"""locations""",1638,"""silver_locations.parquet""",0.04
"""prices""",1198547,"""silver_prices.parquet""",3.95
"""nutritionals""",9467978,"""silver_nutritionals.parquet""",11.43
"""weekly_products""",114517,"""silver_weekly_products.parquet""",7.81
"""weekly_locations""",1230,"""silver_weekly_locations.parque…",0.05
"""weekly_prices""",19075099,"""silver_weekly_prices.parquet""",378.34
"""product_category_paths""",154760,"""silver_product_category_paths.…",7.87
"""product_primary_categories""",167077,"""silver_product_primary_categor…",6.52


In [87]:
# Review persisted observation-level quality metrics without repeating transformations.
display(
    show_metrics(
        "nutritionals",
        [
            "comparable_energy_observations",
            "energy_mismatch_observations",
            "negative_source_nutrient_rows",
        ],
    )
)
display(
    show_metrics(
        "weekly_prices",
        ["multiple_price_version_rows", "multiple_price_version_grains"],
    )
)

metric,value
str,str
"""comparable_energy_observations""","""1081872"""
"""energy_mismatch_observations""","""0"""
"""negative_source_nutrient_rows""","""0"""


metric,value
str,str
"""multiple_price_version_rows""","""751824"""
"""multiple_price_version_grains""","""375912"""


In [88]:
# Close the read-only analytical connection after all evidence has been displayed.
duck.close()
print("Silver checkpoint reviewed. Continue with explicit Gold modelling decisions.")

Silver checkpoint reviewed. Continue with explicit Gold modelling decisions.
